In [ ]:
# =============================================================
# CELL 1: Environment Check (cross-platform)
# Uses PyTorch + nibabel + numpy + matplotlib
# =============================================================
import subprocess
import sys

# Ensure nibabel is available in this kernel.
try:
    import nibabel as nib
except ImportError:
    print("[INFO] nibabel not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nibabel"])
    import nibabel as nib

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA visible: {torch.cuda.is_available()}")

USE_CUDA = False
CUDA_FALLBACK_REASON = ""
if torch.cuda.is_available():
    try:
        x = torch.zeros(1, 1, 8, 8, device='cuda')
        w = torch.zeros(1, 1, 3, 3, device='cuda')
        _ = F.conv2d(x, w, padding=1)
        torch.cuda.synchronize()
        USE_CUDA = True
    except Exception as e:
        CUDA_FALLBACK_REASON = str(e)

device = torch.device('cuda' if USE_CUDA else 'cpu')
print(f"Runtime device: {device}")
if USE_CUDA:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.cuda.is_available():
    print("[WARN] CUDA detected but this GPU is incompatible with the current PyTorch CUDA build.")
    print(f"[WARN] Falling back to CPU. Reason: {CUDA_FALLBACK_REASON}")

print(f"NumPy: {np.__version__}")
print(f"Nibabel: {nib.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print("\n[OK] Required libraries are available.")

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# =============================================================
# CELL 2: Real Data Dataset & DataLoader (PNG images from darren2020/ct-to-mri-cgan)
# CT images: /kaggle/input/ct-to-mri-cgan/images/trainA/
# MRI images: /kaggle/input/ct-to-mri-cgan/images/trainB/
# =============================================================
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T


def strict_minmax_01(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Strict per-sample min-max scaling to [0, 1] for intensity consistency."""
    x_min = x.amin(dim=(-2, -1), keepdim=True)
    x_max = x.amax(dim=(-2, -1), keepdim=True)
    return (x - x_min) / (x_max - x_min + eps)


class CTMRIImageDataset(Dataset):
    """
    Loads paired CT (domain A) and MRI (domain B) PNG images.
    Sorts both lists to align pairs by filename index.
    """
    def __init__(self, ct_dir, mri_dir, image_size=128):
        self.ct_paths  = sorted(glob.glob(os.path.join(ct_dir,  '*.jpg')) +
                                glob.glob(os.path.join(ct_dir,  '*.png')))
        self.mri_paths = sorted(glob.glob(os.path.join(mri_dir, '*.jpg')) +
                                glob.glob(os.path.join(mri_dir, '*.png')))
        # Match lengths
        n = min(len(self.ct_paths), len(self.mri_paths))
        self.ct_paths  = self.ct_paths[:n]
        self.mri_paths = self.mri_paths[:n]
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.Grayscale(num_output_channels=1),
            T.ToTensor(),           # [0,1] float32
        ])
        print(f'[Dataset] {n} paired CT/MRI images loaded.')

    def __len__(self):
        return len(self.ct_paths)

    def __getitem__(self, idx):
        ct  = Image.open(self.ct_paths[idx]).convert('RGB')
        mri = Image.open(self.mri_paths[idx]).convert('RGB')
        ct_t = strict_minmax_01(self.transform(ct))
        mri_t = strict_minmax_01(self.transform(mri))
        return {'ct': ct_t, 'mri': mri_t}

print('[OK] CTMRIImageDataset defined - ready to load real PNG data.')

In [ ]:
# =============================================================
# CELL 3: All Imports (Pure PyTorch - no MONAI required)
# AutoencoderKL (VAE) implemented from scratch using PyTorch
# =============================================================
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt


def resolve_runtime_device():
    """Return (device, use_cuda, reason) after a tiny CUDA kernel smoke test."""
    if not torch.cuda.is_available():
        return torch.device('cpu'), False, 'CUDA not available'

    try:
        x = torch.zeros(1, 1, 8, 8, device='cuda')
        w = torch.zeros(1, 1, 3, 3, device='cuda')
        _ = F.conv2d(x, w, padding=1)
        torch.cuda.synchronize()
        return torch.device('cuda'), True, ''
    except Exception as e:
        return torch.device('cpu'), False, str(e)


device, USE_CUDA, CUDA_FALLBACK_REASON = resolve_runtime_device()
print(f'Device selected: {device}')
if not USE_CUDA and torch.cuda.is_available():
    print('[WARN] CUDA detected but unusable for this PyTorch build. Falling back to CPU.')
    print(f'[WARN] Reason: {CUDA_FALLBACK_REASON}')
print(f'PyTorch: {torch.__version__}')
print('[OK] All imports successful - no MONAI dependency.')

In [ ]:
# =============================================================
# CELL 4: Real Data Setup - dataset discovery (Kaggle + local)
# =============================================================
import os
import glob
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Portable output directory for saved artifacts
if 'OUTPUT_DIR' not in globals():
    if os.path.isdir('/kaggle/working/outputs'):
        OUTPUT_DIR = '/kaggle/working/outputs'
    else:
        OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
        os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')


def find_dataset_path():
    candidates = [
        '/kaggle/input/ct-to-mri-cgan/images',
        '/kaggle/input/datasets/darren2020/ct-to-mri-cgan/Dataset/images',
        '/kaggle/input/ct-to-mri-cgan/Dataset/images',
        os.path.join(os.getcwd(), 'images'),
        os.path.join(os.getcwd(), 'Dataset', 'images'),
        os.path.join(os.getcwd(), 'data', 'images'),
    ]
    for p in candidates:
        if os.path.exists(os.path.join(p, 'trainA')) and os.path.exists(os.path.join(p, 'trainB')):
            return p

    for root, dirs, _ in os.walk(os.getcwd()):
        if 'trainA' in dirs and 'trainB' in dirs:
            return root
    return None


DATA_ROOT = find_dataset_path()
DATA_AVAILABLE = DATA_ROOT is not None

if DATA_AVAILABLE:
    print(f'Dataset found at: {DATA_ROOT}')
    CT_TRAIN_DIR = os.path.join(DATA_ROOT, 'trainA')
    MRI_TRAIN_DIR = os.path.join(DATA_ROOT, 'trainB')

    ct_files = sorted(glob.glob(os.path.join(CT_TRAIN_DIR, '*.jpg')) +
                      glob.glob(os.path.join(CT_TRAIN_DIR, '*.png')))
    mri_files = sorted(glob.glob(os.path.join(MRI_TRAIN_DIR, '*.jpg')) +
                       glob.glob(os.path.join(MRI_TRAIN_DIR, '*.png')))

    if len(ct_files) == 0 or len(mri_files) == 0:
        DATA_AVAILABLE = False
        print('[WARN] trainA/trainB found but no .jpg/.png images were detected.')
    else:
        print(f'CT  (trainA): {len(ct_files)} images')
        print(f'MRI (trainB): {len(mri_files)} images')

if DATA_AVAILABLE:
    n_show = min(5, len(ct_files), len(mri_files))
    fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
    fig.suptitle('Real CT (top) vs MRI (bottom) Brain Scans', fontsize=14)
    for i in range(n_show):
        ct_img = np.array(Image.open(ct_files[i]).convert('L'))
        mri_img = np.array(Image.open(mri_files[i]).convert('L'))
        axes[0, i].imshow(ct_img, cmap='gray'); axes[0, i].set_title(f'CT {i+1}'); axes[0, i].axis('off')
        axes[1, i].imshow(mri_img, cmap='gray'); axes[1, i].set_title(f'MRI {i+1}'); axes[1, i].axis('off')
    plt.tight_layout()
    preview_path = os.path.join(OUTPUT_DIR, 'real_data_samples.png')
    plt.savefig(preview_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'[OK] Real data loaded and samples visualized: {preview_path}')
else:
    print('[WARN] Could not find trainA/trainB image folders. A synthetic fallback dataset will be used in Cell 5.')

In [ ]:
# =============================================================
# CELL 5: Create DataLoader from real or synthetic CT/MRI data
# =============================================================
import torch
from torch.utils.data import DataLoader, Dataset

print('Preparing training dataset...')


def strict_minmax_01(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Strict per-sample min-max scaling to [0,1]."""
    x_min = x.amin(dim=(-2, -1), keepdim=True)
    x_max = x.amax(dim=(-2, -1), keepdim=True)
    return ((x - x_min) / (x_max - x_min + eps)).clamp(0.0, 1.0)


class SyntheticCTMRIDataset(Dataset):
    """Fallback paired dataset so notebook remains runnable without external files."""
    def __init__(self, n_samples=512, image_size=128):
        self.n_samples = n_samples
        self.image_size = image_size

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        base = torch.rand(1, self.image_size, self.image_size)
        ct = torch.clamp(base * 0.7 + 0.2 * torch.rand_like(base), 0, 1)
        mri = torch.clamp(0.6 * ct + 0.4 * torch.rand_like(base), 0, 1)
        ct = strict_minmax_01(ct)
        mri = strict_minmax_01(mri)
        return {'ct': ct.float(), 'mri': mri.float()}


if 'DATA_AVAILABLE' in globals() and DATA_AVAILABLE:
    print(f'Using real data:\n  CT dir:  {CT_TRAIN_DIR}\n  MRI dir: {MRI_TRAIN_DIR}')
    train_ds = CTMRIImageDataset(ct_dir=CT_TRAIN_DIR, mri_dir=MRI_TRAIN_DIR, image_size=128)
else:
    print('[WARN] Real data not available. Using synthetic fallback dataset for code validation.')
    train_ds = SyntheticCTMRIDataset(n_samples=512, image_size=128)

if 'USE_CUDA' not in globals():
    USE_CUDA = False

# NOTE: keep workers=0 in notebooks to avoid multiprocessing parent_pid crashes.
NUM_WORKERS = 0
BATCH_SIZE = 16

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=USE_CUDA,
    drop_last=True,
)
print(f'DataLoader ready: {len(train_ds)} images, {len(train_loader)} batches/epoch')
print(f'Batch size: {BATCH_SIZE}, GPU runtime enabled: {USE_CUDA}, Workers: {NUM_WORKERS}')

batch = next(iter(train_loader))
print(f'CT batch shape: {batch["ct"].shape}')
print(f'MRI batch shape: {batch["mri"].shape}')
print(f'Pixel range CT:  [{batch["ct"].min():.3f}, {batch["ct"].max():.3f}]')
print(f'Pixel range MRI: [{batch["mri"].min():.3f}, {batch["mri"].max():.3f}]')

In [ ]:
# CELL 6 [PATCHED]: AutoencoderKL -- Wider + Attention bottleneck
# KEY FIXES for PSNR:
#   1. ch=64 -> ch=96  (more capacity)
#   2. SelfAttention2D at encoder/decoder bottleneck
#   3. encode() returns deterministic mean (stable latent targets for bridge)
#   4. Bilinear upsample (no checkerboard artifacts)
import torch
import torch.nn as nn
import torch.nn.functional as F

if 'USE_CUDA' not in globals():
    USE_CUDA = torch.cuda.is_available()
device = torch.device('cuda' if USE_CUDA else 'cpu')

class ResBlock(nn.Module):
    def __init__(self, channels, norm_groups=32):
        super().__init__()
        ng = min(norm_groups, channels)
        while channels % ng != 0:
            ng //= 2
        self.block = nn.Sequential(
            nn.GroupNorm(ng, channels), nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(ng, channels), nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )
    def forward(self, x):
        return x + self.block(x)

class SelfAttention2D(nn.Module):
    def __init__(self, channels, norm_groups=32):
        super().__init__()
        ng = min(norm_groups, channels)
        while channels % ng != 0:
            ng //= 2
        self.norm  = nn.GroupNorm(ng, channels)
        self.qkv   = nn.Conv2d(channels, channels * 3, 1)
        self.proj  = nn.Conv2d(channels, channels, 1)
        self.scale = channels ** -0.5
    def forward(self, x):
        B, C, H, W = x.shape
        h   = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.softmax(torch.bmm(q.permute(0, 2, 1), k) * self.scale, dim=-1)
        out  = torch.bmm(v, attn.permute(0, 2, 1)).reshape(B, C, H, W)
        return x + self.proj(out)

class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False))

class VAEEncoder(nn.Module):
    def __init__(self, in_ch=1, ch=96, ch_mult=(1, 2, 4), latent_ch=4, n_res=2):
        super().__init__()
        self.conv_in = nn.Conv2d(in_ch, ch, 3, padding=1)
        channels = [ch * m for m in ch_mult]
        blocks = []
        prev = ch
        for i, c in enumerate(channels):
            blocks.append(nn.Conv2d(prev, c, 3, padding=1))
            for _ in range(n_res):
                blocks.append(ResBlock(c))
            if i == len(channels) - 1:
                blocks.append(SelfAttention2D(c))
            if i < len(channels) - 1:
                blocks.append(Downsample(c))
            prev = c
        self.blocks = nn.Sequential(*blocks)
        ng = 32
        while prev % ng != 0:
            ng //= 2
        self.norm_out = nn.GroupNorm(ng, prev)
        self.conv_out = nn.Conv2d(prev, 2 * latent_ch, 1)
    def forward(self, x):
        x = self.conv_in(x)
        x = self.blocks(x)
        x = F.silu(self.norm_out(x))
        return self.conv_out(x)

class VAEDecoder(nn.Module):
    def __init__(self, out_ch=1, ch=96, ch_mult=(1, 2, 4), latent_ch=4, n_res=2):
        super().__init__()
        channels = [ch * m for m in reversed(ch_mult)]
        self.conv_in = nn.Conv2d(latent_ch, channels[0], 3, padding=1)
        blocks = []
        prev = channels[0]
        for i, c in enumerate(channels):
            if i > 0:
                blocks.append(nn.Conv2d(prev, c, 3, padding=1))
                prev = c
            if i == 0:
                blocks.append(SelfAttention2D(c))
            for _ in range(n_res):
                blocks.append(ResBlock(c))
            if i < len(channels) - 1:
                blocks.append(Upsample(c))
        self.blocks = nn.Sequential(*blocks)
        ng = 32
        while prev % ng != 0:
            ng //= 2
        self.norm_out = nn.GroupNorm(ng, prev)
        self.conv_out = nn.Conv2d(prev, out_ch, 3, padding=1)
    def forward(self, x):
        x = self.conv_in(x)
        x = self.blocks(x)
        x = F.silu(self.norm_out(x))
        return torch.sigmoid(self.conv_out(x))

class AutoencoderKL(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, ch=96, ch_mult=(1, 2, 4), latent_ch=4, n_res=2):
        super().__init__()
        self.encoder  = VAEEncoder(in_ch, ch, ch_mult, latent_ch, n_res)
        self.decoder  = VAEDecoder(out_ch, ch, ch_mult, latent_ch, n_res)
        self.latent_ch = latent_ch

    def encode(self, x):
        # Returns deterministic mean -- stable supervision for bridge
        h = self.encoder(x)
        mean, logvar = h.chunk(2, dim=1)
        logvar = torch.clamp(logvar, -30, 20)
        return mean, (mean, logvar)

    def encode_stochastic(self, x):
        # Stochastic encode used only during VAE training
        h = self.encoder(x)
        mean, logvar = h.chunk(2, dim=1)
        logvar = torch.clamp(logvar, -30, 20)
        std = torch.exp(0.5 * logvar)
        z   = mean + std * torch.randn_like(mean)
        return z, (mean, logvar)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z, (mean, logvar) = self.encode_stochastic(x)
        return self.decode(z), mean, logvar

autoencoder = AutoencoderKL(
    in_ch=1, out_ch=1, ch=96, ch_mult=(1, 2, 4), latent_ch=4, n_res=2
).to(device)
total_params = sum(p.numel() for p in autoencoder.parameters())
print(f'AutoencoderKL [PATCHED] on {device}  |  Params: {total_params/1e6:.2f}M')
with torch.no_grad():
    dummy = torch.zeros(2, 1, 128, 128).to(device)
    recon, mean, logvar = autoencoder(dummy)
    print(f'Input: {dummy.shape} -> Recon: {recon.shape}, Mean: {mean.shape}')
print('[OK] Patched AutoencoderKL ready.')


In [ ]:
# CELL 7 [PATCHED]: VAE Training -- SSIM loss + proper schedule
# KEY FIXES:
#   1. 50 epochs (was 20)
#   2. Added SSIM loss (1 - SSIM minimised directly)
#   3. Discriminator warmup: adv_weight=0 for first 5 epochs
#   4. CosineAnnealingWarmRestarts
#   5. Tuned L1=4.0, SSIM=2.0, ADV_max=0.1
import os
import numpy as np
from torchvision import models
from tqdm.auto import tqdm

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def sobel_edge_map(x):
    c = x.shape[1]
    kx = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=x.dtype, device=x.device).view(1,1,3,3).repeat(c,1,1,1)
    ky = torch.tensor([[1,2,1],[0,0,0],[-1,-2,-1]], dtype=x.dtype, device=x.device).view(1,1,3,3).repeat(c,1,1,1)
    return torch.sqrt(F.conv2d(x, kx, padding=1, groups=c)**2 + F.conv2d(x, ky, padding=1, groups=c)**2 + 1e-8)

def sobel_edge_loss(pred, target):
    return F.l1_loss(sobel_edge_map(pred), sobel_edge_map(target))

def spectral_convergence_loss(pred, target, eps=1e-8):
    p = torch.nan_to_num(pred.float()); t = torch.nan_to_num(target.float())
    pm = torch.abs(torch.fft.rfft2(p, norm='ortho'))
    tm = torch.abs(torch.fft.rfft2(t, norm='ortho'))
    num = torch.norm((pm - tm).reshape(pred.shape[0], -1), dim=1)
    den = torch.norm(tm.reshape(target.shape[0], -1), dim=1).clamp_min(eps)
    return torch.nan_to_num((num / den).mean(), nan=0.0, posinf=1e3)

def ssim_loss(pred, target, ws=11):
    C1, C2 = 0.01**2, 0.03**2; pad = ws // 2
    mu1 = F.avg_pool2d(pred, ws, stride=1, padding=pad)
    mu2 = F.avg_pool2d(target, ws, stride=1, padding=pad)
    mu1_sq, mu2_sq = mu1*mu1, mu2*mu2
    s1 = F.avg_pool2d(pred*pred, ws, stride=1, padding=pad) - mu1_sq
    s2 = F.avg_pool2d(target*target, ws, stride=1, padding=pad) - mu2_sq
    s12 = F.avg_pool2d(pred*target, ws, stride=1, padding=pad) - mu1*mu2
    return 1 - ((2*mu1*mu2 + C1)*(2*s12 + C2) / ((mu1_sq+mu2_sq+C1)*(s1+s2+C2))).mean()

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_channels=1, base_filters=64, n_layers=3):
        super().__init__()
        kw, padw = 4, 1
        seq = [nn.Conv2d(in_channels, base_filters, kw, stride=2, padding=padw), nn.LeakyReLU(0.2, True)]
        nf = 1
        for n in range(1, n_layers):
            nf_p = nf; nf = min(2**n, 8)
            seq += [nn.Conv2d(base_filters*nf_p, base_filters*nf, kw, stride=2, padding=padw, bias=False),
                    nn.BatchNorm2d(base_filters*nf), nn.LeakyReLU(0.2, True)]
        nf_p = nf; nf = min(2**n_layers, 8)
        seq += [nn.Conv2d(base_filters*nf_p, base_filters*nf, kw, stride=1, padding=padw, bias=False),
                nn.BatchNorm2d(base_filters*nf), nn.LeakyReLU(0.2, True),
                nn.Conv2d(base_filters*nf, 1, kw, stride=1, padding=padw)]
        self.model = nn.Sequential(*seq)
    def forward(self, x): return self.model(x)

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        self.slice = nn.Sequential(*list(vgg.children())[:16]).eval()
        for p in self.parameters(): p.requires_grad = False
    def forward(self, x, y):
        x3 = x.repeat(1,3,1,1); y3 = y.repeat(1,3,1,1)
        return F.mse_loss(self.slice(x3), self.slice(y3))

discriminator       = PatchGANDiscriminator(in_channels=1).to(device)
optimizer_d         = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.9))
perceptual_criterion = PerceptualLoss().to(device)

VAE_EPOCHS        = 50
LEARNING_RATE     = 1e-4
L1_WEIGHT         = 4.0
SSIM_WEIGHT       = 2.0
KL_WEIGHT         = 1e-6
PERC_WEIGHT       = 0.5
ADV_WEIGHT_MAX    = 0.1
ADV_WARMUP_EPOCHS = 5
EDGE_WEIGHT       = 1.0
FFT_WEIGHT        = 0.3
SAVE_PATH_VAE     = os.path.join(OUTPUT_DIR, 'autoencoder_mri.pth')
SAVE_PATH_DISC    = os.path.join(OUTPUT_DIR, 'discriminator_mri.pth')

optimizer_vae = torch.optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)
scheduler_vae = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_vae, T_0=10, T_mult=2)

print(f'VAE training [PATCHED] for {VAE_EPOCHS} epochs on {device}')
print(f'L1={L1_WEIGHT} SSIM={SSIM_WEIGHT} EDGE={EDGE_WEIGHT} FFT={FFT_WEIGHT} PERC={PERC_WEIGHT}')

vae_history = []; best_vae_psnr = -1.0

for epoch in range(VAE_EPOCHS):
    autoencoder.train(); discriminator.train()
    running_loss = running_d_loss = running_psnr = 0.0
    valid_batches = skipped = 0
    adv_weight = ADV_WEIGHT_MAX * max(0, epoch - ADV_WARMUP_EPOCHS) / max(1, VAE_EPOCHS - ADV_WARMUP_EPOCHS)

    pbar = tqdm(train_loader, desc=f'VAE {epoch+1}/{VAE_EPOCHS}')
    for batch in pbar:
        imgs = batch['mri'].to(device).clamp(0, 1)
        imgs = torch.nan_to_num(imgs)

        optimizer_d.zero_grad(set_to_none=True)
        with torch.no_grad():
            recon_det, _, _ = autoencoder(imgs)
        pred_real = discriminator(imgs)
        pred_fake = discriminator(recon_det.detach())
        loss_d = 0.5 * (F.binary_cross_entropy_with_logits(pred_real, torch.ones_like(pred_real)) +
                        F.binary_cross_entropy_with_logits(pred_fake, torch.zeros_like(pred_fake)))
        if not torch.isfinite(loss_d): skipped += 1; continue
        loss_d.backward()
        nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
        optimizer_d.step()

        optimizer_vae.zero_grad(set_to_none=True)
        recon, mu, logvar = autoencoder(imgs)
        recon = torch.nan_to_num(recon).clamp(0, 1)
        l1_l   = F.l1_loss(recon, imgs)
        ssim_l = ssim_loss(recon, imgs)
        edge_l = sobel_edge_loss(recon, imgs)
        fft_l  = spectral_convergence_loss(recon, imgs)
        perc_l = perceptual_criterion(recon, imgs)
        kl_l   = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / imgs.size(0)
        adv_l  = F.binary_cross_entropy_with_logits(discriminator(recon), torch.ones_like(discriminator(recon)))
        total_loss = (L1_WEIGHT*l1_l + SSIM_WEIGHT*ssim_l + EDGE_WEIGHT*edge_l +
                      FFT_WEIGHT*fft_l + PERC_WEIGHT*perc_l + KL_WEIGHT*kl_l + adv_weight*adv_l)
        if not torch.isfinite(total_loss): skipped += 1; continue
        total_loss.backward()
        nn.utils.clip_grad_norm_(autoencoder.parameters(), 1.0)
        optimizer_vae.step()

        with torch.no_grad():
            mse_b  = F.mse_loss(recon, imgs).item()
            psnr_b = 20 * np.log10(1.0 / (np.sqrt(max(mse_b, 1e-8)) + 1e-8))
        valid_batches  += 1
        running_loss   += total_loss.item()
        running_d_loss += loss_d.item()
        running_psnr   += psnr_b
        pbar.set_postfix({'G': f'{total_loss.item():.4f}', 'PSNR': f'{psnr_b:.1f}', 'SSIM_l': f'{ssim_l.item():.4f}'})

    scheduler_vae.step()
    denom = max(1, valid_batches)
    avg_psnr = running_psnr / denom
    vae_history.append((running_loss / denom, avg_psnr))
    torch.save(autoencoder.state_dict(), SAVE_PATH_VAE)
    torch.save(discriminator.state_dict(), SAVE_PATH_DISC)
    if avg_psnr > best_vae_psnr:
        best_vae_psnr = avg_psnr
        torch.save(autoencoder.state_dict(), SAVE_PATH_VAE.replace('.pth', '_best.pth'))
    print(f'[Epoch {epoch+1}/{VAE_EPOCHS}] PSNR={avg_psnr:.2f}dB adv_w={adv_weight:.4f} skip={skipped}')

print(f'[OK] VAE done. Best PSNR: {best_vae_psnr:.2f} dB')
best_path = SAVE_PATH_VAE.replace('.pth', '_best.pth')
if os.path.exists(best_path):
    autoencoder.load_state_dict(torch.load(best_path, map_location=device))
    print('[OK] Best VAE weights loaded.')


In [ ]:
# =============================================================
# CELL 8: VAE Reconstruction Visualization + CLAHE processing
# =============================================================
from skimage import exposure
import matplotlib.pyplot as plt
import numpy as np

autoencoder.eval()

batch = next(iter(train_loader))
mri_orig = batch['mri'][:4].to(device)

with torch.no_grad():
    mri_recon, mean, logvar = autoencoder(mri_orig)
    mri_recon = torch.nan_to_num(mri_recon, nan=0.0, posinf=1.0, neginf=0.0).clamp(0, 1)


def psnr(a, b):
    mse = F.mse_loss(a, b).item()
    if not np.isfinite(mse) or mse <= 0:
        return 99.0
    return 20 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))


psnr_val = psnr(mri_recon, mri_orig)
print(f'PSNR (VAE reconstruction): {psnr_val:.2f} dB')

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle(f'MRI Autoencoder Reconstruction & Normalization (PSNR={psnr_val:.1f} dB)', fontsize=14)

for i in range(4):
    orig = mri_orig[i, 0].detach().cpu().numpy()
    recon = mri_recon[i, 0].detach().cpu().numpy()

    recon_clamped = np.clip(recon, 0, 1)
    recon_clahe = exposure.equalize_adapthist(recon_clamped, clip_limit=0.03)

    axes[0, i].imshow(orig, cmap='gray')
    axes[0, i].set_title(f'Original MRI {i+1}')
    axes[0, i].axis('off')

    axes[1, i].imshow(recon_clamped, cmap='gray')
    axes[1, i].set_title(f'VAE Recon {i+1}')
    axes[1, i].axis('off')

    axes[2, i].imshow(recon_clahe, cmap='gray')
    axes[2, i].set_title(f'Recon + CLAHE {i+1}')
    axes[2, i].axis('off')

plt.tight_layout()

try:
    OUTPUT_DIR_CLEAN = OUTPUT_DIR
except NameError:
    OUTPUT_DIR_CLEAN = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR_CLEAN, exist_ok=True)

save_path = os.path.join(OUTPUT_DIR_CLEAN, 'vae_reconstruction_clahe.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'[OK] VAE reconstruction visualization saved to {save_path}.')

In [ ]:
# CELL 9 [PATCHED]: CTtoLatentUNet -- Skip connections + wider channels
# KEY FIXES:
#   1. Skip connections (classic UNet) -- preserves spatial detail
#   2. base_ch=64 (was 32/64 no-skip)
#   3. Residual blocks at each level
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.block(x) + self.skip(x)

class CTtoLatentUNet(nn.Module):
    def __init__(self, in_ch=1, latent_ch=4, base_ch=64):
        super().__init__()
        bc = base_ch
        self.enc1      = ConvResBlock(in_ch, bc)
        self.pool1     = nn.MaxPool2d(2)
        self.enc2      = ConvResBlock(bc, bc*2)
        self.pool2     = nn.MaxPool2d(2)
        self.bottleneck = ConvResBlock(bc*2, bc*4)
        self.up1       = nn.ConvTranspose2d(bc*4, bc*2, 2, stride=2)
        self.dec1      = ConvResBlock(bc*4, bc*2)
        self.up2       = nn.ConvTranspose2d(bc*2, bc, 2, stride=2)
        self.dec2      = ConvResBlock(bc*2, bc)
        self.to_latent = nn.Sequential(
            nn.Conv2d(bc, bc, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(bc, bc//2, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(bc//2, latent_ch, 1),
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b  = self.bottleneck(self.pool2(e2))
        d1 = self.dec1(torch.cat([self.up1(b), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d1), e1], dim=1))
        return self.to_latent(d2)

ct2latent = CTtoLatentUNet(in_ch=1, latent_ch=4, base_ch=64).to(device)
total_g = sum(p.numel() for p in ct2latent.parameters())
print(f'CTtoLatentUNet [PATCHED] on {device}  |  Params: {total_g/1e6:.2f}M')
with torch.no_grad():
    dummy_ct = torch.zeros(2, 1, 128, 128).to(device)
    z_pred = ct2latent(dummy_ct)
print(f'CT input: {dummy_ct.shape} -> Latent output: {z_pred.shape}')
print('[OK] Patched CTtoLatentUNet ready.')


In [ ]:
# ============================================================
# CELL 10b [REPLACED]: Stage 2 Bridge Training (PSNR-focused)
# Objective: L_lat + lambda_mse * L_mse + lambda_l1 * L_l1
# No perceptual or SSIM terms for bridge optimization.
# ============================================================
import os
import time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# Execution-order safety: define metrics helpers here if Cell 13 has not run yet.
if 'compute_psnr' not in globals():
    def compute_psnr(pred, gt):
        pred = pred.float().clamp(0.0, 1.0)
        gt = gt.float().clamp(0.0, 1.0)
        mse = F.mse_loss(pred, gt).item()
        if not np.isfinite(mse) or mse <= 0:
            return 99.0
        return 20.0 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))

if 'compute_ssim_simple' not in globals():
    def compute_ssim_simple(img1, img2):
        x = img1.float().clamp(0.0, 1.0)
        y = img2.float().clamp(0.0, 1.0)
        c1, c2 = 0.01 ** 2, 0.03 ** 2
        mu1, mu2 = x.mean(), y.mean()
        var1 = ((x - mu1) ** 2).mean()
        var2 = ((y - mu2) ** 2).mean()
        cov12 = ((x - mu1) * (y - mu2)).mean()
        ssim = ((2 * mu1 * mu2 + c1) * (2 * cov12 + c2)) / ((mu1 ** 2 + mu2 ** 2 + c1) * (var1 + var2 + c2) + 1e-8)
        return float(ssim.item())

if 'eval_bridge_metrics' not in globals():
    @torch.no_grad()
    def eval_bridge_metrics(loader, max_batches=20):
        ct2latent.eval()
        autoencoder.eval()
        psnr_vals, ssim_vals = [], []

        for bi, batch in enumerate(loader):
            if bi >= max_batches:
                break
            ct = batch['ct'].to(device).float().clamp(0, 1)
            mri = batch['mri'].to(device).float().clamp(0, 1)
            z_pred = ct2latent(ct)
            mri_pred = autoencoder.decode(z_pred).clamp(0, 1)
            psnr_vals.append(compute_psnr(mri_pred, mri))
            ssim_vals.append(compute_ssim_simple(mri_pred, mri))

        return {
            'psnr': float(np.mean(psnr_vals)) if psnr_vals else 0.0,
            'ssim': float(np.mean(ssim_vals)) if ssim_vals else 0.0,
        }

BRIDGE_EPOCHS = 100
LR_BRIDGE = 2e-4
VAL_SPLIT = 0.12
EARLY_STOP_PATIENCE = 12
VAL_MAX_BATCHES = 25
# Optional speed control for notebooks: set env BRIDGE_MAX_TRAIN_BATCHES to limit batches/epoch.
BRIDGE_MAX_TRAIN_BATCHES = int(os.environ.get('BRIDGE_MAX_TRAIN_BATCHES', '0'))

LAMBDA_MSE = 1.0
LAMBDA_L1 = 0.5

BRIDGE_BEST = os.path.join(OUTPUT_DIR, 'ct2latent_best.pth')
BRIDGE_LAST = os.path.join(OUTPUT_DIR, 'ct2latent_last.pth')
BRIDGE_CKPT = os.path.join(OUTPUT_DIR, 'ct2latent_bridge_ckpt.pth')

# Build train/val split for bridge if not already present.
if 'train_loader_bridge' in globals() and 'val_loader_bridge' in globals():
    print('[Bridge] Reusing existing train_loader_bridge / val_loader_bridge.')
else:
    base_ds = train_loader.dataset
    n_total = len(base_ds)
    n_val = max(1, int(n_total * VAL_SPLIT))
    n_train = max(1, n_total - n_val)
    split_gen = torch.Generator().manual_seed(42)
    ds_train, ds_val = random_split(base_ds, [n_train, n_val], generator=split_gen)

    bs = getattr(train_loader, 'batch_size', 8)
    nw = getattr(train_loader, 'num_workers', 0)
    pm = getattr(train_loader, 'pin_memory', False)

    train_loader_bridge = DataLoader(ds_train, batch_size=bs, shuffle=True, num_workers=nw, pin_memory=pm, drop_last=True)
    val_loader_bridge = DataLoader(ds_val, batch_size=bs, shuffle=False, num_workers=nw, pin_memory=pm, drop_last=False)
    print(f'[Bridge] Created split: train={n_train}, val={n_val}')

# Freeze VAE, train bridge only.
autoencoder.eval()
for p in autoencoder.parameters():
    p.requires_grad = False
for p in ct2latent.parameters():
    p.requires_grad = True

optimizer_bridge = torch.optim.AdamW(ct2latent.parameters(), lr=LR_BRIDGE, weight_decay=1e-4)
use_amp = torch.cuda.is_available()
scaler_bridge = torch.amp.GradScaler(enabled=use_amp)

start_epoch = 0
best_val_psnr = -1.0
epochs_no_improve = 0

if os.path.exists(BRIDGE_CKPT):
    ckpt = torch.load(BRIDGE_CKPT, map_location=device)
    ct2latent.load_state_dict(ckpt['model'])
    optimizer_bridge.load_state_dict(ckpt['optimizer'])
    start_epoch = int(ckpt.get('epoch', -1)) + 1
    best_val_psnr = float(ckpt.get('best_val_psnr', -1.0))
    epochs_no_improve = int(ckpt.get('epochs_no_improve', 0))
    print(f'[Bridge RESUME] epoch={start_epoch}, best_val_psnr={best_val_psnr:.2f} dB')
elif os.path.exists(BRIDGE_BEST):
    ct2latent.load_state_dict(torch.load(BRIDGE_BEST, map_location=device))
    print('[Bridge] Loaded existing best checkpoint.')

print(f'[Bridge] Training for up to {BRIDGE_EPOCHS} epochs | lr={LR_BRIDGE}')
print(f'[Bridge] Loss = LatentL1 + {LAMBDA_MSE}*ImageMSE + {LAMBDA_L1}*ImageL1')
if BRIDGE_MAX_TRAIN_BATCHES > 0:
    print(f'[Bridge] Speed mode: max {BRIDGE_MAX_TRAIN_BATCHES} train batches per epoch')

for epoch in range(start_epoch, BRIDGE_EPOCHS):
    ct2latent.train()
    loss_sum = 0.0
    psnr_sum = 0.0
    nb = 0

    pbar = tqdm(train_loader_bridge, desc=f'Bridge {epoch + 1}/{BRIDGE_EPOCHS}')
    for bi, batch in enumerate(pbar):
        if BRIDGE_MAX_TRAIN_BATCHES > 0 and bi >= BRIDGE_MAX_TRAIN_BATCHES:
            break

        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri = batch['mri'].to(device).float().clamp(0, 1)

        optimizer_bridge.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            with torch.no_grad():
                # Deterministic mean latent supervision (no sampling).
                z_gt, _ = autoencoder.encode(mri)

            z_pred = ct2latent(ct)
            mri_pred = autoencoder.decode(z_pred).clamp(0, 1)

            l_lat = F.l1_loss(z_pred, z_gt)
            l_mse = F.mse_loss(mri_pred, mri)
            l_l1 = F.l1_loss(mri_pred, mri)
            loss = l_lat + LAMBDA_MSE * l_mse + LAMBDA_L1 * l_l1

        scaler_bridge.scale(loss).backward()
        scaler_bridge.unscale_(optimizer_bridge)
        torch.nn.utils.clip_grad_norm_(ct2latent.parameters(), 1.0)
        scaler_bridge.step(optimizer_bridge)
        scaler_bridge.update()

        with torch.no_grad():
            psnr_b = compute_psnr(mri_pred, mri)

        loss_sum += float(loss.item())
        psnr_sum += float(psnr_b)
        nb += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'psnr': f'{psnr_b:.2f}'})

    train_loss = loss_sum / max(1, nb)
    train_psnr = psnr_sum / max(1, nb)

    val_stats = eval_bridge_metrics(val_loader_bridge, max_batches=VAL_MAX_BATCHES)
    val_psnr = val_stats['psnr']

    print(f'[Bridge {epoch+1}] train_loss={train_loss:.4f} train_psnr={train_psnr:.2f}dB val_psnr={val_psnr:.2f}dB')

    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        epochs_no_improve = 0
        torch.save(ct2latent.state_dict(), BRIDGE_BEST)
        print(f'  [BEST] Saved {BRIDGE_BEST} | val_psnr={best_val_psnr:.2f} dB')
    else:
        epochs_no_improve += 1

    torch.save(ct2latent.state_dict(), BRIDGE_LAST)
    torch.save(
        {
            'epoch': epoch,
            'model': ct2latent.state_dict(),
            'optimizer': optimizer_bridge.state_dict(),
            'best_val_psnr': best_val_psnr,
            'epochs_no_improve': epochs_no_improve,
        },
        BRIDGE_CKPT,
    )

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'[EARLY STOP] No val PSNR improvement for {EARLY_STOP_PATIENCE} epochs.')
        break

if os.path.exists(BRIDGE_BEST):
    ct2latent.load_state_dict(torch.load(BRIDGE_BEST, map_location=device))
print(f'[OK] Bridge done. Best val PSNR = {best_val_psnr:.2f} dB')

In [ ]:
# ============================================================
# CELL 10a-utils [REPLACED]: Shared Metrics and Evaluation Helpers
# NOTE: Cross-modality CT->MRI has an intrinsic PSNR ceiling.
# Realistic bridge target is typically around 18-20 dB; diffusion is
# optimized for anatomical realism and downstream classifier utility,
# so its PSNR is reported as a secondary metric.
# ============================================================
import os
import numpy as np
import torch
import torch.nn.functional as F

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

if 'compute_psnr' not in globals():
    def compute_psnr(pred, gt):
        pred = pred.float().clamp(0.0, 1.0)
        gt = gt.float().clamp(0.0, 1.0)
        mse = F.mse_loss(pred, gt).item()
        if not np.isfinite(mse) or mse <= 0:
            return 99.0
        return 20.0 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))

if 'compute_ssim_simple' not in globals():
    def compute_ssim_simple(img1, img2):
        x = img1.float().clamp(0.0, 1.0)
        y = img2.float().clamp(0.0, 1.0)
        c1, c2 = 0.01 ** 2, 0.03 ** 2
        mu1, mu2 = x.mean(), y.mean()
        var1 = ((x - mu1) ** 2).mean()
        var2 = ((y - mu2) ** 2).mean()
        cov12 = ((x - mu1) * (y - mu2)).mean()
        ssim = ((2 * mu1 * mu2 + c1) * (2 * cov12 + c2)) / ((mu1 ** 2 + mu2 ** 2 + c1) * (var1 + var2 + c2) + 1e-8)
        return float(ssim.item())

@torch.no_grad()
def eval_bridge_metrics(loader, max_batches=20):
    ct2latent.eval()
    autoencoder.eval()
    psnr_vals, ssim_vals = [], []

    for bi, batch in enumerate(loader):
        if bi >= max_batches:
            break
        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri = batch['mri'].to(device).float().clamp(0, 1)

        z_pred = ct2latent(ct)
        mri_pred = autoencoder.decode(z_pred).clamp(0, 1)

        psnr_vals.append(compute_psnr(mri_pred, mri))
        ssim_vals.append(compute_ssim_simple(mri_pred, mri))

    return {
        'psnr': float(np.mean(psnr_vals)) if psnr_vals else 0.0,
        'ssim': float(np.mean(ssim_vals)) if ssim_vals else 0.0,
    }

print('[OK] Shared metrics loaded: compute_psnr, compute_ssim_simple, eval_bridge_metrics')

In [ ]:
# ============================================================
# CELL 10c [REPLACED]: Stage 2 Evaluation + generative_metrics.csv
# Uses shared compute_psnr for VAE/Bridge/Diffusion.
# Diffusion PSNR/SSIM are reported as secondary metrics.
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

BRIDGE_BEST = os.path.join(OUTPUT_DIR, 'ct2latent_best.pth')
DDPM_BEST = os.path.join(OUTPUT_DIR, 'cond_unet_ddpm_best.pth')

if os.path.exists(BRIDGE_BEST):
    ct2latent.load_state_dict(torch.load(BRIDGE_BEST, map_location=device))
    print(f'[Eval] Loaded bridge best checkpoint: {BRIDGE_BEST}')
else:
    print('[Eval] Bridge best checkpoint not found, using current bridge weights.')

if 'cond_unet' in globals() and os.path.exists(DDPM_BEST):
    cond_unet.load_state_dict(torch.load(DDPM_BEST, map_location=device))
    cond_unet.eval()
    print(f'[Eval] Loaded diffusion best checkpoint: {DDPM_BEST}')

loader_eval = val_loader_bridge if 'val_loader_bridge' in globals() else (val_loader if 'val_loader' in globals() else train_loader)

autoencoder.eval()
ct2latent.eval()
if 'cond_unet' in globals():
    cond_unet.eval()

@torch.no_grad()
def sample_diffusion_eval(ct_batch):
    if 'ddpm_sample_fast' in globals():
        return ddpm_sample_fast(ct_batch, n_steps=globals().get('DDPM_SAMPLE_STEPS', 50), guidance_scale=2.0).clamp(0, 1)
    # Fallback if DDPM sampler is unavailable: bridge decode.
    z_b = ct2latent(ct_batch)
    return autoencoder.decode(z_b).clamp(0, 1)

vae_psnr, vae_ssim = [], []
bridge_psnr, bridge_ssim = [], []
diff_psnr, diff_ssim = [], []

qual_ct = qual_mri = qual_vae = qual_bridge = qual_diff = None

MAX_EVAL_BATCHES = 10
with torch.no_grad():
    for bi, batch in enumerate(loader_eval):
        if bi >= MAX_EVAL_BATCHES:
            break

        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri = batch['mri'].to(device).float().clamp(0, 1)

        z_gt, _ = autoencoder.encode(mri)
        mri_vae = autoencoder.decode(z_gt).clamp(0, 1)

        z_bridge = ct2latent(ct)
        mri_bridge = autoencoder.decode(z_bridge).clamp(0, 1)

        mri_diff = sample_diffusion_eval(ct)

        vae_psnr.append(compute_psnr(mri_vae, mri))
        vae_ssim.append(compute_ssim_simple(mri_vae, mri))

        bridge_psnr.append(compute_psnr(mri_bridge, mri))
        bridge_ssim.append(compute_ssim_simple(mri_bridge, mri))

        diff_psnr.append(compute_psnr(mri_diff, mri))
        diff_ssim.append(compute_ssim_simple(mri_diff, mri))

        if qual_ct is None:
            n_show = min(4, ct.shape[0])
            qual_ct = ct[:n_show].detach().cpu()
            qual_mri = mri[:n_show].detach().cpu()
            qual_vae = mri_vae[:n_show].detach().cpu()
            qual_bridge = mri_bridge[:n_show].detach().cpu()
            qual_diff = mri_diff[:n_show].detach().cpu()

metrics_rows = [
    {'stage': 'vae_reconstruction', 'psnr_db': round(float(np.mean(vae_psnr)) if vae_psnr else 0.0, 3), 'ssim': round(float(np.mean(vae_ssim)) if vae_ssim else 0.0, 5)},
    {'stage': 'bridge_ct_to_mri', 'psnr_db': round(float(np.mean(bridge_psnr)) if bridge_psnr else 0.0, 3), 'ssim': round(float(np.mean(bridge_ssim)) if bridge_ssim else 0.0, 5)},
    {'stage': 'diffusion_ct_to_mri', 'psnr_db': round(float(np.mean(diff_psnr)) if diff_psnr else 0.0, 3), 'ssim': round(float(np.mean(diff_ssim)) if diff_ssim else 0.0, 5)},
]

metrics_df = pd.DataFrame(metrics_rows)
metrics_path = os.path.join(OUTPUT_DIR, 'generative_metrics.csv')
metrics_df.to_csv(metrics_path, index=False)

print('=' * 70)
print('GENERATION METRICS (shared compute_psnr)')
print('=' * 70)
for row in metrics_rows:
    stage = row['stage']
    extra = ' [secondary for diffusion]' if stage == 'diffusion_ct_to_mri' else ''
    print(f"{stage:<28} | PSNR={row['psnr_db']:.3f} dB | SSIM={row['ssim']:.5f}{extra}")
print('=' * 70)
print(f'Saved metrics: {metrics_path}')

# Qualitative grid: CT | Real MRI | VAE recon | Bridge MRI | Diffusion MRI
if qual_ct is not None:
    n = qual_ct.shape[0]
    fig, axes = plt.subplots(n, 5, figsize=(18, 3.5 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    titles = ['Input CT', 'Real MRI', 'VAE Recon', 'Bridge MRI', 'Synth MRI (Diffusion)']
    for j, t in enumerate(titles):
        axes[0, j].set_title(t, fontsize=11)

    for i in range(n):
        axes[i, 0].imshow(qual_ct[i, 0].numpy(), cmap='bone')
        axes[i, 1].imshow(qual_mri[i, 0].numpy(), cmap='gray')
        axes[i, 2].imshow(qual_vae[i, 0].numpy(), cmap='gray')
        axes[i, 3].imshow(qual_bridge[i, 0].numpy(), cmap='gray')
        axes[i, 4].imshow(qual_diff[i, 0].numpy(), cmap='gray')
        for j in range(5):
            axes[i, j].axis('off')

    fig.suptitle('Stage-2 Generation Comparison (Diffusion metrics are secondary)', fontsize=13)
    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, 'stage2_bridge_eval.jpg')
    plt.savefig(fig_path, dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Saved qualitative panel: {fig_path}')

In [ ]:
# =============================================================
# CELL 10c [REPLACED]: SHARPNESS GAP VISUALIZATION (unified metrics)
# Key fix: bridge fallback now decodes RAW bridge latents (no /LATENT_SCALE).
# =============================================================
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import torch.nn.functional as F

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

if 'LATENT_SCALE' not in globals():
    LATENT_SCALE = 0.18215

if 'compute_psnr' not in globals():
    def compute_psnr(pred, gt):
        pred = pred.float().clamp(0, 1)
        gt = gt.float().clamp(0, 1)
        mse = F.mse_loss(pred, gt).item()
        if not np.isfinite(mse) or mse <= 0:
            return 99.0
        return 20.0 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))

if 'compute_ssim_simple' not in globals():
    def compute_ssim_simple(img1, img2):
        x = img1.float().clamp(0, 1)
        y = img2.float().clamp(0, 1)
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        mu1, mu2 = x.mean(), y.mean()
        var1 = ((x - mu1) ** 2).mean()
        var2 = ((y - mu2) ** 2).mean()
        cov12 = ((x - mu1) * (y - mu2)).mean()
        ssim = ((2 * mu1 * mu2 + C1) * (2 * cov12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (var1 + var2 + C2) + 1e-8)
        return float(ssim.item())

def to_np(t):
    t = t.detach().cpu().float()
    if t.dim() == 4:
        t = t[0, 0]
    elif t.dim() == 3:
        t = t[0]
    return t.clamp(0, 1).numpy()

autoencoder.eval()
ct2latent.eval()
if 'cond_unet' in globals():
    cond_unet.eval()

N_VIZ = 4
viz_loader = bridge_val_loader if 'bridge_val_loader' in globals() else (val_loader if 'val_loader' in globals() else train_loader)
batch = next(iter(viz_loader))
ct_viz = batch['ct'][:N_VIZ].to(device).float().clamp(0, 1)
mri_viz = batch['mri'][:N_VIZ].to(device).float().clamp(0, 1)

with torch.no_grad():
    z_gt, _ = autoencoder.encode(mri_viz)
    mri_recon = autoencoder.decode(z_gt).clamp(0, 1)

    if 'cond_unet' in globals() and callable(globals().get('ddpm_sample', None)):
        mri_diffusion = ddpm_sample(ct_viz, n_steps=50, guidance_scale=2.0).clamp(0, 1)
        col4_title = 'Diffusion Output\n(CT -> DDPM -> MRI)'
        monitor_title = 'Sharpness Gap Monitor - True Diffusion Output'
        mode_note = 'DDPM mode'
    else:
        z_bridge = ct2latent(ct_viz)
        # FIX: bridge fallback is RAW latent decode, not /LATENT_SCALE.
        mri_diffusion = autoencoder.decode(z_bridge).clamp(0, 1)
        col4_title = 'Bridge Fallback\n(CT -> Latent -> Decode)'
        monitor_title = 'Sharpness Gap Monitor (Fallback)' 
        mode_note = 'Fallback mode: cond_unet/ddpm_sample unavailable'

fig = plt.figure(figsize=(16, 4 * N_VIZ))
fig.patch.set_facecolor('#0d0d0d')

col_titles = [
    'Input CT',
    'Real MRI\n(Ground Truth)',
    'VAE Reconstruction\n(Upper Bound)',
    col4_title,
]
col_cmaps = ['bone', 'gray', 'gray', 'gray']

gs = gridspec.GridSpec(
    N_VIZ + 1,
    4,
    figure=fig,
    hspace=0.08,
    wspace=0.05,
    height_ratios=[0.35] + [1] * N_VIZ,
)

for c, title in enumerate(col_titles):
    ax = fig.add_subplot(gs[0, c])
    ax.text(0.5, 0.5, title, transform=ax.transAxes, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    ax.axis('off')
    ax.set_facecolor('#0d0d0d')

for i in range(N_VIZ):
    imgs = [
        to_np(ct_viz[i:i + 1]),
        to_np(mri_viz[i:i + 1]),
        to_np(mri_recon[i:i + 1]),
        to_np(mri_diffusion[i:i + 1]),
    ]

    psnr_recon = compute_psnr(mri_recon[i:i + 1], mri_viz[i:i + 1])
    psnr_diff = compute_psnr(mri_diffusion[i:i + 1], mri_viz[i:i + 1])
    ssim_recon = compute_ssim_simple(mri_recon[i:i + 1], mri_viz[i:i + 1])
    ssim_diff = compute_ssim_simple(mri_diffusion[i:i + 1], mri_viz[i:i + 1])

    for c, (img, cmap) in enumerate(zip(imgs, col_cmaps)):
        ax = fig.add_subplot(gs[i + 1, c])
        ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
        ax.axis('off')
        ax.set_facecolor('#0d0d0d')

        if c == 2:
            ax.set_title(f'PSNR {psnr_recon:.1f}dB | SSIM {ssim_recon:.3f}', fontsize=7.5, color='#aaffaa', pad=3)
        elif c == 3:
            gap = psnr_recon - psnr_diff
            color = '#aaffaa' if gap < 2 else ('#ffcc66' if gap < 5 else '#ff6666')
            ax.set_title(f'PSNR {psnr_diff:.1f}dB | SSIM {ssim_diff:.3f} [gap {gap:+.1f}dB]', fontsize=7.5, color=color, pad=3)

        if c == 0:
            ax.set_ylabel(f'Sample {i + 1}', fontsize=9, color='#aaaaaa', labelpad=6)

fig.suptitle(f'{monitor_title}\n{mode_note}', fontsize=13, fontweight='bold', color='white', y=1.01)

SAVE_PATH_VIZ = os.path.join(OUTPUT_DIR, 'sharpness_gap_monitor.png')
plt.savefig(SAVE_PATH_VIZ, dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
plt.close()

print('\n' + '=' * 62)
print(f"{'Sample':<10} {'VAE PSNR':>10} {'Diff PSNR':>11} {'Gap':>8} {'VAE SSIM':>10} {'Diff SSIM':>11}")
print('-' * 62)
for i in range(N_VIZ):
    pr = compute_psnr(mri_recon[i:i + 1], mri_viz[i:i + 1])
    pd = compute_psnr(mri_diffusion[i:i + 1], mri_viz[i:i + 1])
    sr = compute_ssim_simple(mri_recon[i:i + 1], mri_viz[i:i + 1])
    sd = compute_ssim_simple(mri_diffusion[i:i + 1], mri_viz[i:i + 1])
    status = 'SHARP' if (pr - pd) < 2 else ('CLOSE' if (pr - pd) < 5 else 'BLURRY')
    print(f"{'Sample ' + str(i + 1):<10} {pr:>10.2f} {pd:>11.2f} {pr - pd:>+8.2f} {sr:>10.4f} {sd:>11.4f}  [{status}]")
print('=' * 62)
print(f'Saved -> {SAVE_PATH_VIZ}')

In [ ]:
# =============================================================
# CELL 14 [UPDATED]: Phase 4 - Full Pipeline Evaluation & Visualization
# Uses shared compute_psnr for consistency with bridge/debug/sharpness cells.
# Robust checkpoint loading: prefers best checkpoints, falls back to legacy names.
# =============================================================
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torch.nn.functional as F

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

if 'compute_psnr' not in globals():
    def compute_psnr(pred, gt):
        pred = pred.float().clamp(0, 1)
        gt = gt.float().clamp(0, 1)
        mse = F.mse_loss(pred, gt).item()
        if not np.isfinite(mse) or mse <= 0:
            return 99.0
        return 20.0 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))


def load_first_existing(model, candidate_paths, model_label):
    for p in candidate_paths:
        if os.path.exists(p):
            model.load_state_dict(torch.load(p, map_location=device))
            print(f'[OK] Loaded {model_label}: {p}')
            return p
    print(f'[WARN] No checkpoint found for {model_label}; using current in-memory weights.')
    return None

vae_candidates = [
    os.path.join(OUTPUT_DIR, 'autoencoder_mri_best.pth'),
    os.path.join(OUTPUT_DIR, 'autoencoder_mri.pth'),
]
bridge_candidates = [
    os.path.join(OUTPUT_DIR, 'ct2latent_best.pth'),
    os.path.join(OUTPUT_DIR, 'ct2latent_last.pth'),
    os.path.join(OUTPUT_DIR, 'ct2latent.pth'),  # legacy
]

print('Loading trained models...')
vae_path_loaded = load_first_existing(autoencoder, vae_candidates, 'autoencoder')
bridge_path_loaded = load_first_existing(ct2latent, bridge_candidates, 'ct2latent bridge')
autoencoder.eval()
ct2latent.eval()
print('[OK] Model loading step complete.')

test_loader = val_loader_bridge if 'val_loader_bridge' in globals() else (val_loader if 'val_loader' in globals() else train_loader)
test_batch = next(iter(test_loader))
ct_sample = test_batch['ct'][:4].to(device).float().clamp(0, 1)
mri_gt = test_batch['mri'][:4].to(device).float().clamp(0, 1)

with torch.no_grad():
    z_pred = ct2latent(ct_sample)
    mri_synth = autoencoder.decode(z_pred).clamp(0, 1)
    z_gt, _ = autoencoder.encode(mri_gt)
    mri_upper = autoencoder.decode(z_gt).clamp(0, 1)

psnr_synth = compute_psnr(mri_synth, mri_gt)
psnr_upper = compute_psnr(mri_upper, mri_gt)
l1_synth = F.l1_loss(mri_synth, mri_gt).item()

gap = psnr_upper - psnr_synth
print('\n--- Pipeline Metrics ---')
print(f'PSNR (CT->Synth MRI vs GT):   {psnr_synth:.2f} dB')
print(f'PSNR (VAE recon upper bound): {psnr_upper:.2f} dB')
print(f'L1   (CT->Synth MRI vs GT):   {l1_synth:.4f}')
print(f'Gap to upper bound:           {gap:.2f} dB')

n = 4
fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
fig.suptitle(
    f'CT -> Synthetic MRI  |  PSNR={psnr_synth:.1f} dB  (Upper={psnr_upper:.1f} dB)',
    fontsize=14,
    fontweight='bold',
)
col_titles = ['CT Input', 'Synthetic MRI (Bridge)', 'Real MRI (Ground Truth)', 'VAE Upper Bound']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11)

for i in range(n):
    axes[i, 0].imshow(ct_sample[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 1].imshow(mri_synth[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 2].imshow(mri_gt[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 3].imshow(mri_upper[i, 0].cpu().numpy(), cmap='gray')
    for j in range(4):
        axes[i, j].axis('off')

out_img = os.path.join(OUTPUT_DIR, 'ct_to_mri_results.png')
plt.tight_layout()
plt.savefig(out_img, dpi=120, bbox_inches='tight')
plt.show()

print('\n[OK] Evaluation complete!')
print(f'Results saved: {out_img}')
print(f'Loaded VAE checkpoint: {vae_path_loaded}')
print(f'Loaded Bridge checkpoint: {bridge_path_loaded}')

In [ ]:
# =============================================================
# CELL 15 [NEW]: DirectCTtoMRIUNet - Direct Pixel-Space Baseline Model
# Simple 2D U-Net for CT->MRI mapping without VAE/diffusion.
# =============================================================
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class DirectCTtoMRIUNet(nn.Module):
    """4-level U-Net: CT (1ch) -> MRI (1ch), both 128x128 grayscale in [0,1]."""

    def __init__(self, base_ch=32):
        super().__init__()
        self.enc1 = ConvBlock(1, base_ch)            # 32
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_ch, 2 * base_ch)  # 64
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(2 * base_ch, 4 * base_ch)  # 128
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = ConvBlock(4 * base_ch, 8 * base_ch)  # 256
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(8 * base_ch, 8 * base_ch)  # 256

        self.up4 = UpBlock(8 * base_ch, 8 * base_ch, 4 * base_ch)  # 256 + 256 -> 128
        self.up3 = UpBlock(4 * base_ch, 4 * base_ch, 2 * base_ch)  # 128 + 128 -> 64
        self.up2 = UpBlock(2 * base_ch, 2 * base_ch, base_ch)      # 64 + 64 -> 32
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 2, stride=2)

        self.final = nn.Sequential(
            ConvBlock(2 * base_ch, base_ch),
            nn.Conv2d(base_ch, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool1(s1))
        s3 = self.enc3(self.pool2(s2))
        s4 = self.enc4(self.pool3(s3))

        b = self.bottleneck(self.pool4(s4))

        d4 = self.up4(b, s4)
        d3 = self.up3(d4, s3)
        d2 = self.up2(d3, s2)
        d1 = self.up1(d2)

        out = self.final(torch.cat([d1, s1], dim=1))
        return out


# Instantiate model
ct2mri_unet = DirectCTtoMRIUNet(base_ch=32).to(device)
print(f'[OK] DirectCTtoMRIUNet created: {sum(p.numel() for p in ct2mri_unet.parameters()) / 1e6:.1f}M params')

In [ ]:
# =============================================================
# CELL 16 [NEW]: Train DirectCTtoMRIUNet with L1+MSE+SSIM Loss
# Validation split (10%), early stopping on val PSNR.
# =============================================================
import os
import numpy as np
import torch
import torch.optim as optim
from torch.amp import autocast, GradScaler
import torch.utils.data as tdata

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

if 'compute_psnr' not in globals():
    import torch.nn.functional as F

    def compute_psnr(pred, gt):
        pred = pred.float().clamp(0, 1)
        gt = gt.float().clamp(0, 1)
        mse = F.mse_loss(pred, gt).item()
        if not np.isfinite(mse) or mse <= 0:
            return 99.0
        return 20.0 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))

if 'compute_ssim_simple' not in globals():
    def compute_ssim_simple(img1, img2):
        x = img1.float().clamp(0, 1)
        y = img2.float().clamp(0, 1)
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        mu1, mu2 = x.mean(), y.mean()
        var1 = ((x - mu1) ** 2).mean()
        var2 = ((y - mu2) ** 2).mean()
        cov12 = ((x - mu1) * (y - mu2)).mean()
        ssim = ((2 * mu1 * mu2 + C1) * (2 * cov12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (var1 + var2 + C2) + 1e-8)
        return float(ssim.item())


# Split existing dataset into train/val for U-Net baseline
split_idx = int(0.9 * len(train_loader.dataset))
train_subset, val_subset = tdata.random_split(
    train_loader.dataset,
    [split_idx, len(train_loader.dataset) - split_idx],
    generator=torch.Generator().manual_seed(42),
)

bs = getattr(train_loader, 'batch_size', 8)
nw = getattr(train_loader, 'num_workers', 0)
pm = getattr(train_loader, 'pin_memory', False)

train_loader_unet = tdata.DataLoader(train_subset, batch_size=bs, shuffle=True, num_workers=nw, pin_memory=pm, drop_last=True)
val_loader_unet = tdata.DataLoader(val_subset, batch_size=bs, shuffle=False, num_workers=nw, pin_memory=pm, drop_last=False)
print(f'[INFO] U-Net split ready: train={len(train_subset)} val={len(val_subset)}')


def ssim_loss(img1, img2):
    img1 = img1.float().clamp(0, 1)
    img2 = img2.float().clamp(0, 1)
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    mu1 = img1.mean()
    mu2 = img2.mean()
    var1 = ((img1 - mu1) ** 2).mean()
    var2 = ((img2 - mu2) ** 2).mean()
    cov12 = ((img1 - mu1) * (img2 - mu2)).mean()
    ssim_val = ((2 * mu1 * mu2 + C1) * (2 * cov12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (var1 + var2 + C2) + 1e-8)
    return 1.0 - ssim_val


optimizer = optim.AdamW(ct2mri_unet.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=1)

use_amp = torch.cuda.is_available()
scaler = GradScaler('cuda', enabled=use_amp)

epochs = 100
patience = 10
best_val_psnr = -np.inf
best_epoch = 0
patience_counter = 0
UNET_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'ct2mri_unet_best.pth')

print('\n=== Training DirectCTtoMRIUNet ===')
for epoch in range(epochs):
    ct2mri_unet.train()
    train_loss, train_psnr, train_ssim = 0.0, 0.0, 0.0

    for batch in train_loader_unet:
        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri = batch['mri'].to(device).float().clamp(0, 1)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=use_amp):
            pred = ct2mri_unet(ct)
            l1_l = torch.nn.functional.l1_loss(pred, mri)
            mse_l = torch.nn.functional.mse_loss(pred, mri)
            ssim_l = ssim_loss(pred, mri)
            loss = 0.5 * l1_l + 1.0 * mse_l + 0.5 * ssim_l

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(ct2mri_unet.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += float(loss.item())
        with torch.no_grad():
            train_psnr += compute_psnr(pred, mri)
            train_ssim += compute_ssim_simple(pred, mri)

    scheduler.step()
    train_loss /= max(1, len(train_loader_unet))
    train_psnr /= max(1, len(train_loader_unet))
    train_ssim /= max(1, len(train_loader_unet))

    ct2mri_unet.eval()
    val_loss, val_psnr, val_ssim = 0.0, 0.0, 0.0
    with torch.no_grad():
        for batch in val_loader_unet:
            ct = batch['ct'].to(device).float().clamp(0, 1)
            mri = batch['mri'].to(device).float().clamp(0, 1)
            pred = ct2mri_unet(ct)

            l1_l = torch.nn.functional.l1_loss(pred, mri)
            mse_l = torch.nn.functional.mse_loss(pred, mri)
            ssim_l = ssim_loss(pred, mri)
            loss = 0.5 * l1_l + 1.0 * mse_l + 0.5 * ssim_l

            val_loss += float(loss.item())
            val_psnr += compute_psnr(pred, mri)
            val_ssim += compute_ssim_simple(pred, mri)

    val_loss /= max(1, len(val_loader_unet))
    val_psnr /= max(1, len(val_loader_unet))
    val_ssim /= max(1, len(val_loader_unet))

    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        best_epoch = epoch
        patience_counter = 0
        torch.save(ct2mri_unet.state_dict(), UNET_CHECKPOINT_PATH)
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch < 5:
        print(f'Epoch {epoch + 1:3d} | TrainLoss {train_loss:.4f} | Val PSNR {val_psnr:.2f} dB | Val SSIM {val_ssim:.4f}')

    if patience_counter >= patience:
        print(f'[EARLY STOP] No improvement for {patience} epochs. Best val PSNR: {best_val_psnr:.2f} dB @ epoch {best_epoch + 1}')
        break

print(f'[OK] Training complete. Best checkpoint: {UNET_CHECKPOINT_PATH}')

In [ ]:
# =============================================================
# CELL 17 [NEW]: Evaluate DirectCTtoMRIUNet, Update Metrics CSV & Visualization
# Loads best checkpoint, evaluates on test set, appends to generative_metrics.csv
# =============================================================
import torch
import torch.nn as nn
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

if 'compute_psnr' not in globals():
    import torch.nn.functional as F
    def compute_psnr(pred, gt):
        pred = pred.float().clamp(0, 1)
        gt = gt.float().clamp(0, 1)
        mse = F.mse_loss(pred, gt).item()
        if not np.isfinite(mse) or mse <= 0:
            return 99.0
        return 20.0 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))

if 'compute_ssim_simple' not in globals():
    def compute_ssim_simple(img1, img2):
        x = img1.float().clamp(0, 1)
        y = img2.float().clamp(0, 1)
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        mu1, mu2 = x.mean(), y.mean()
        var1 = ((x - mu1) ** 2).mean()
        var2 = ((y - mu2) ** 2).mean()
        cov12 = ((x - mu1) * (y - mu2)).mean()
        ssim = ((2 * mu1 * mu2 + C1) * (2 * cov12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (var1 + var2 + C2) + 1e-8)
        return float(ssim.item())

# Load best U-Net checkpoint
UNET_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'ct2mri_unet_best.pth')
ct2mri_unet.eval()
if os.path.exists(UNET_CHECKPOINT_PATH):
    ct2mri_unet.load_state_dict(torch.load(UNET_CHECKPOINT_PATH, map_location=device))
    print(f'[OK] Loaded U-Net checkpoint: {UNET_CHECKPOINT_PATH}')
else:
    print(f'[WARN] No checkpoint found at {UNET_CHECKPOINT_PATH}; using in-memory weights.')

# Evaluate on test set
test_loader_eval = val_loader if 'val_loader' in globals() else train_loader
test_psnr_list, test_ssim_list = [], []

with torch.no_grad():
    for batch in test_loader_eval:
        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri_gt = batch['mri'].to(device).float().clamp(0, 1)
        pred = ct2mri_unet(ct)
        
        for i in range(pred.shape[0]):
            test_psnr_list.append(compute_psnr(pred[i:i+1], mri_gt[i:i+1]))
            test_ssim_list.append(compute_ssim_simple(pred[i:i+1], mri_gt[i:i+1]))

mean_psnr = np.mean(test_psnr_list)
mean_ssim = np.mean(test_ssim_list)
std_psnr = np.std(test_psnr_list)
std_ssim = np.std(test_ssim_list)

print('\n=== DirectCTtoMRIUNet Test Results ===')
print(f'Mean PSNR: {mean_psnr:.2f} ± {std_psnr:.2f} dB')
print(f'Mean SSIM: {mean_ssim:.4f} ± {std_ssim:.4f}')

# Update generative_metrics.csv
metrics_csv = os.path.join(OUTPUT_DIR, 'generative_metrics.csv')
unet_row = pd.DataFrame([{
    'stage': 'direct_unet_ct_to_mri',
    'psnr_db': round(mean_psnr, 2),
    'ssim': round(mean_ssim, 4),
}])

if os.path.exists(metrics_csv):
    df = pd.read_csv(metrics_csv)
    # Remove existing U-Net row if present
    df = df[df['stage'] != 'direct_unet_ct_to_mri']
    df = pd.concat([df, unet_row], ignore_index=True)
else:
    df = unet_row

df.to_csv(metrics_csv, index=False)
print(f'\n[OK] Updated metrics CSV: {metrics_csv}')
print(df.to_string(index=False))

# Optional: Generate comparison figure
n_samples = 4
sample_indices = np.linspace(0, len(test_loader_eval.dataset) - 1, n_samples, dtype=int)

fig, axes = plt.subplots(n_samples, 5, figsize=(20, 4 * n_samples))
fig.suptitle('CT->MRI Comparison: Direct U-Net vs. Rest of Pipeline', fontsize=14, fontweight='bold')

col_titles = ['CT Input', 'Real MRI', 'VAE Recon', 'Bridge->Diffusion', 'Direct U-Net']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=10)

sample_batch = next(iter(test_loader_eval))
ct_batch = sample_batch['ct'][:n_samples].to(device).float().clamp(0, 1)
mri_batch = sample_batch['mri'][:n_samples].to(device).float().clamp(0, 1)

with torch.no_grad():
    # Get other outputs
    if 'autoencoder' in globals():
        z_gt, _ = autoencoder.encode(mri_batch)
        mri_vae_recon = autoencoder.decode(z_gt).clamp(0, 1)
    else:
        mri_vae_recon = mri_batch
    
    if 'ct2latent' in globals():
        z_bridge = ct2latent(ct_batch)
        mri_bridge = autoencoder.decode(z_bridge).clamp(0, 1)
    else:
        mri_bridge = mri_batch * 0.5  # fallback
    
    # Direct U-Net
    mri_unet = ct2mri_unet(ct_batch)

for i in range(n_samples):
    axes[i, 0].imshow(ct_batch[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 1].imshow(mri_batch[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 2].imshow(mri_vae_recon[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 3].imshow(mri_bridge[i, 0].cpu().numpy(), cmap='gray')
    axes[i, 4].imshow(mri_unet[i, 0].cpu().numpy(), cmap='gray')
    for j in range(5):
        axes[i, j].axis('off')

unet_fig_path = os.path.join(OUTPUT_DIR, 'direct_unet_ct_to_mri_results.jpg')
plt.tight_layout()
plt.savefig(unet_fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'[OK] Comparison figure saved: {unet_fig_path}')


In [ ]:
# =============================================================
# CELL 11: Conditioned U-Net with Cross-Attention for DDPM
# FIX 1: All upsampling uses Upsample(bilinear) + Conv2d
#         (eliminates checkerboard artifacts from ConvTranspose2d)
# FIX 2: LATENT_SCALE constant defined here for use in Cell 12
# Paper Spec: U-Net denoising with CT latent conditioning via
# cross-attention: Attention(Q, K, V) where K,V from CT encoder
# Latent space: 4-channel, spatial 32x32 (from 128->32 VAE)
# =============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Latent scaling factor (keeps latent variance near 1 for stable diffusion)
LATENT_SCALE = 0.18215

# --- Sinusoidal Timestep Embedding ---
class SinusoidalPE(nn.Module):
    """Sinusoidal positional encoding for diffusion timesteps."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=device) / (half - 1)
        )
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return emb  # (B, dim)

# --- Cross-Attention Block: Attention(Q, K, V) ---
# Q from noisy MRI latent, K and V from CT conditioning latent
class CrossAttention(nn.Module):
    def __init__(self, query_dim, context_dim, heads=4, dim_head=32):
        super().__init__()
        inner_dim = heads * dim_head
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        self.to_k = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_out = nn.Linear(inner_dim, query_dim)

    def forward(self, x, context):
        B, N, C = x.shape
        _, M, _ = context.shape
        h = self.heads
        q = self.to_q(x).reshape(B, N, h, -1).permute(0, 2, 1, 3)
        k = self.to_k(context).reshape(B, M, h, -1).permute(0, 2, 1, 3)
        v = self.to_v(context).reshape(B, M, h, -1).permute(0, 2, 1, 3)
        attn = torch.softmax(torch.einsum('bhid,bhjd->bhij', q, k) * self.scale, dim=-1)
        out = torch.einsum('bhij,bhjd->bhid', attn, v)
        out = out.permute(0, 2, 1, 3).reshape(B, N, -1)
        return self.to_out(out)

# --- Transformer Block with Self + Cross Attention ---
class TransformerBlock(nn.Module):
    def __init__(self, channels, context_dim, heads=4, dim_head=32):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.norm3 = nn.LayerNorm(channels)
        self.self_attn = CrossAttention(channels, channels, heads, dim_head)
        self.cross_attn = CrossAttention(channels, context_dim, heads, dim_head)
        self.ff = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels * 4),
            nn.GELU(),
            nn.Linear(channels * 4, channels),
        )

    def forward(self, x, context):
        B, C, H, W = x.shape
        x_flat = x.reshape(B, C, H * W).permute(0, 2, 1)   # (B, HW, C)
        # Self-attention
        x_norm = self.norm1(x_flat)
        sa_out, _ = self.self_attn(x_norm, x_norm, x_norm) if False else (self.self_attn(x_norm, x_norm), None)
        x_flat = x_flat + sa_out
        # Cross-attention: Q=x, K/V=context
        x_flat = x_flat + self.cross_attn(self.norm2(x_flat), context)
        # Feed-forward
        x_flat = x_flat + self.ff(x_flat)
        return x_flat.permute(0, 2, 1).reshape(B, C, H, W)

# --- UNet Block (Down or Up) ---
# FIX: Upsampling uses nn.Upsample (bilinear) + Conv2d — NO ConvTranspose2d
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, context_dim, upsample=False, downsample=False):
        super().__init__()
        self.upsample = upsample
        self.downsample = downsample

        self.conv1 = nn.Sequential(
            nn.GroupNorm(min(32, in_ch), in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch),
        )
        self.conv2 = nn.Sequential(
            nn.GroupNorm(min(32, out_ch), out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.attn = TransformerBlock(out_ch, context_dim)

        # Spatial resampling — bilinear upsample to avoid checkerboard
        if upsample:
            self.resample = nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
            )
        elif downsample:
            self.resample = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)
        else:
            self.resample = nn.Identity()

    def forward(self, x, t_emb, context):
        h = self.conv1(x)
        # Add time embedding
        t = self.time_proj(t_emb)[:, :, None, None]
        h = h + t
        h = self.conv2(h)
        h = h + self.res_conv(x)
        h = self.attn(h, context)
        h = self.resample(h)
        return h

# --- Conditioned U-Net for Latent Diffusion ---
class CondUNet(nn.Module):
    """
    U-Net denoiser for latent diffusion.
    Input:  noisy latent z_t (B, 4, 32, 32) + CT context (B, 4, 32, 32)
    Output: predicted noise eps (B, 4, 32, 32)
    All upsampling uses bilinear interpolation to prevent checkerboard artifacts.
    """
    def __init__(self, in_ch=4, base_ch=128, time_dim=256, context_ch=4):
        super().__init__()
        self.time_dim = time_dim
        self.time_embed = nn.Sequential(
            SinusoidalPE(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )
        # Context encoder: flatten CT latent (B,4,32,32) -> (B, HW, context_ch)
        # We project context channels to a consistent dim for cross-attention
        context_dim = 64
        self.context_proj = nn.Conv2d(context_ch, context_dim, 1)

        # Input projection
        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        # Encoder (downsampling path)
        self.down1 = UNetBlock(base_ch,     base_ch,     time_dim, context_dim, downsample=False)
        self.down2 = UNetBlock(base_ch,     base_ch * 2, time_dim, context_dim, downsample=True)
        self.down3 = UNetBlock(base_ch * 2, base_ch * 4, time_dim, context_dim, downsample=True)

        # Bottleneck
        self.mid1  = UNetBlock(base_ch * 4, base_ch * 4, time_dim, context_dim)
        self.mid2  = UNetBlock(base_ch * 4, base_ch * 4, time_dim, context_dim)

        # Decoder (upsampling path) — uses bilinear upsample + Conv2d, NO ConvTranspose2d
        self.up3   = UNetBlock(base_ch * 4 + base_ch * 4, base_ch * 4, time_dim, context_dim, upsample=True)
        self.up2   = UNetBlock(base_ch * 4 + base_ch * 2, base_ch * 2, time_dim, context_dim, upsample=True)
        self.up1   = UNetBlock(base_ch * 2 + base_ch,     base_ch,     time_dim, context_dim)

        # Output projection
        self.out_conv = nn.Sequential(
            nn.GroupNorm(min(32, base_ch), base_ch),
            nn.SiLU(),
            nn.Conv2d(base_ch, in_ch, 3, padding=1),
        )

    def forward(self, x, t, ct_context):
        """
        x:          (B, 4, 32, 32)  noisy latent
        t:          (B,)             timestep indices
        ct_context: (B, 4, 32, 32)  CT latent from ct2latent
        """
        # Time embedding
        t_emb = self.time_embed(t)  # (B, time_dim)

        # Context: project CT latent channels, then flatten to sequence
        ctx = self.context_proj(ct_context)                          # (B, 64, 32, 32)
        B, Cc, Hc, Wc = ctx.shape
        context_seq = ctx.reshape(B, Cc, Hc * Wc).permute(0, 2, 1) # (B, HW, 64)

        # Encoder
        x = self.in_conv(x)                        # (B, 128, 32, 32)
        d1 = self.down1(x,  t_emb, context_seq)    # (B, 128, 32, 32)
        d2 = self.down2(d1, t_emb, context_seq)    # (B, 256, 16, 16)
        d3 = self.down3(d2, t_emb, context_seq)    # (B, 512,  8,  8)

        # Bottleneck
        m = self.mid1(d3, t_emb, context_seq)      # (B, 512, 8, 8)
        m = self.mid2(m,  t_emb, context_seq)      # (B, 512, 8, 8)

        # Decoder with skip connections + bilinear upsampling
        u = self.up3(torch.cat([m,  d3], dim=1), t_emb, context_seq)  # (B, 512, 16, 16)
        u = self.up2(torch.cat([u,  d2], dim=1), t_emb, context_seq)  # (B, 256, 32, 32)
        u = self.up1(torch.cat([u,  d1], dim=1), t_emb, context_seq)  # (B, 128, 32, 32)

        return self.out_conv(u)  # (B, 4, 32, 32) — predicted noise

# Instantiate and move to device
cond_unet = CondUNet(
    in_ch=4,
    base_ch=128,
    time_dim=256,
    context_ch=4,
).to(device)

total_params = sum(p.numel() for p in cond_unet.parameters())
print(f'CondUNet parameters: {total_params:,}')
print('[FIX] Upsampling: nn.Upsample(bilinear) + Conv2d (NO ConvTranspose2d)')
print('[FIX] LATENT_SCALE =', LATENT_SCALE)
print('[OK] CondUNet instantiated and moved to device.')


In [ ]:
# ============================================================
# CELL 12 [REPLACED]: Diffusion Training (realism-focused)
# Diffusion is optimized for latent noise prediction and qualitative realism,
# with PSNR treated as a secondary monitoring signal.
# ============================================================
import os
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

if 'LATENT_SCALE' not in globals():
    LATENT_SCALE = 0.18215
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

DDPM_EPOCHS = 120
LR_DDPM = 2e-4
TRAIN_T_STEPS = 200
SAMPLE_T_STEPS = 50
VAL_EVERY = 5
VAL_MAX_BATCHES = 6
EARLY_STOP_PATIENCE = 10

# Keep image-space reconstruction term disabled by default for stability.
REC_LOSS_WEIGHT = 0.0

DDPM_BEST = os.path.join(OUTPUT_DIR, 'cond_unet_ddpm_best.pth')
DDPM_LAST = os.path.join(OUTPUT_DIR, 'cond_unet_ddpm_last.pth')
DDPM_CKPT = os.path.join(OUTPUT_DIR, 'cond_unet_ddpm_ckpt.pth')

# Ensure diffusion is trained after improved bridge and with frozen VAE+bridge.
BRIDGE_BEST = os.path.join(OUTPUT_DIR, 'ct2latent_best.pth')
if os.path.exists(BRIDGE_BEST):
    ct2latent.load_state_dict(torch.load(BRIDGE_BEST, map_location=device))
    print(f'[DDPM] Loaded improved bridge checkpoint: {BRIDGE_BEST}')

autoencoder.eval()
ct2latent.eval()
for p in autoencoder.parameters():
    p.requires_grad = False
for p in ct2latent.parameters():
    p.requires_grad = False
for p in cond_unet.parameters():
    p.requires_grad = True

use_amp = torch.cuda.is_available()
scaler_ddpm = torch.amp.GradScaler(enabled=use_amp)


def cosine_beta_schedule(T, s=0.008):
    steps = torch.arange(T + 1, dtype=torch.float32, device=device)
    ac = torch.cos(((steps / T) + s) / (1 + s) * math.pi / 2) ** 2
    ac = ac / ac[0]
    return torch.clamp(1 - (ac[1:] / ac[:-1]), 1e-4, 0.9999)

betas_train = cosine_beta_schedule(TRAIN_T_STEPS)
alphas_train = 1.0 - betas_train
alphas_cumprod_train = torch.cumprod(alphas_train, dim=0)
alphas_cumprod_prev_train = F.pad(alphas_cumprod_train[:-1], (1, 0), value=1.0)
sqrt_alphas_cumprod_train = torch.sqrt(alphas_cumprod_train)
sqrt_one_minus_alphas_cumprod_train = torch.sqrt(1.0 - alphas_cumprod_train)

# Keep global tensors for later eval/inference cells.
T_STEPS = TRAIN_T_STEPS
betas = betas_train
alphas = alphas_train
alphas_cumprod = alphas_cumprod_train
alphas_cumprod_prev = alphas_cumprod_prev_train
sample_indices = torch.linspace(TRAIN_T_STEPS - 1, 0, SAMPLE_T_STEPS, device=device).long()
DDPM_SAMPLE_STEPS = SAMPLE_T_STEPS


def q_sample_train(z0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(z0)
    sa = sqrt_alphas_cumprod_train[t][:, None, None, None]
    so = sqrt_one_minus_alphas_cumprod_train[t][:, None, None, None]
    return sa * z0 + so * noise, noise


@torch.no_grad()
def ddpm_sample_fast(ct_input, n_steps=SAMPLE_T_STEPS, guidance_scale=2.0):
    cond_unet.eval()
    B = ct_input.size(0)
    z_ct = ct2latent(ct_input) * LATENT_SCALE
    z_uncond = torch.zeros_like(z_ct)
    z = torch.randn(B, 4, 32, 32, device=device)

    idx = sample_indices if n_steps == SAMPLE_T_STEPS else torch.linspace(TRAIN_T_STEPS - 1, 0, n_steps, device=device).long()
    for cur_t in idx:
        t_int = int(cur_t.item())
        t_batch = torch.full((B,), t_int, device=device, dtype=torch.long)

        eps_cond = cond_unet(z, t_batch, z_ct)
        eps_uncond = cond_unet(z, t_batch, z_uncond)
        eps_pred = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        alpha_t = alphas_train[t_int]
        alpha_bar = alphas_cumprod_train[t_int]
        alpha_bp = alphas_cumprod_prev_train[t_int]
        beta_t = betas_train[t_int]

        x0_pred = torch.clamp((z - torch.sqrt(1 - alpha_bar) * eps_pred) / torch.sqrt(alpha_bar), -3, 3)
        coef1 = beta_t * torch.sqrt(alpha_bp) / (1 - alpha_bar)
        coef2 = (1 - alpha_bp) * torch.sqrt(alpha_t) / (1 - alpha_bar)
        mu = coef1 * x0_pred + coef2 * z

        if t_int > 0:
            sigma = torch.sqrt(beta_t * (1 - alpha_bp) / (1 - alpha_bar))
            z = mu + sigma * torch.randn_like(z)
        else:
            z = mu

    return autoencoder.decode(z / LATENT_SCALE).clamp(0, 1)


@torch.no_grad()
def run_ddpm_val_noise_loss(loader, max_batches=VAL_MAX_BATCHES):
    cond_unet.eval()
    vals = []
    for bi, batch in enumerate(loader):
        if bi >= max_batches:
            break
        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri = batch['mri'].to(device).float().clamp(0, 1)

        z0, _ = autoencoder.encode(mri)
        z0 = z0 * LATENT_SCALE
        z_ct = ct2latent(ct) * LATENT_SCALE

        B = ct.size(0)
        t = torch.randint(0, TRAIN_T_STEPS, (B,), device=device, dtype=torch.long)
        z_noisy, noise_gt = q_sample_train(z0, t)
        noise_pred = cond_unet(z_noisy, t, z_ct)
        loss_noise = F.mse_loss(noise_pred, noise_gt)
        vals.append(float(loss_noise.item()))

    return float(np.mean(vals)) if vals else 1e9


optimizer_ddpm = torch.optim.AdamW(cond_unet.parameters(), lr=LR_DDPM, weight_decay=1e-4)
scheduler_ddpm = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_ddpm, T_0=30, T_mult=2)

start_epoch = 0
best_val_noise = 1e9
epochs_no_improve = 0

if os.path.exists(DDPM_CKPT):
    ckpt = torch.load(DDPM_CKPT, map_location=device)
    cond_unet.load_state_dict(ckpt['model'])
    optimizer_ddpm.load_state_dict(ckpt['optimizer'])
    scheduler_ddpm.load_state_dict(ckpt['scheduler'])
    start_epoch = int(ckpt.get('epoch', -1)) + 1
    best_val_noise = float(ckpt.get('best_val_noise', best_val_noise))
    epochs_no_improve = int(ckpt.get('epochs_no_improve', 0))
    print(f'[DDPM RESUME] epoch={start_epoch}, best_val_noise={best_val_noise:.6f}')
elif os.path.exists(DDPM_BEST):
    cond_unet.load_state_dict(torch.load(DDPM_BEST, map_location=device))
    print('[DDPM] Loaded existing best diffusion checkpoint.')

print(f'[DDPM] Training up to {DDPM_EPOCHS} epochs | lr={LR_DDPM} | rec_w={REC_LOSS_WEIGHT}')

loader_val = val_loader_bridge if 'val_loader_bridge' in globals() else (val_loader if 'val_loader' in globals() else train_loader)

for epoch in range(start_epoch, DDPM_EPOCHS):
    cond_unet.train()
    epoch_loss = 0.0
    ns = 0

    pbar = tqdm(train_loader, desc=f'DDPM {epoch+1}/{DDPM_EPOCHS}')
    for batch in pbar:
        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri = batch['mri'].to(device).float().clamp(0, 1)
        B = ct.size(0)

        optimizer_ddpm.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            with torch.no_grad():
                z0, _ = autoencoder.encode(mri)
                z0 = z0 * LATENT_SCALE
                z_ct = ct2latent(ct) * LATENT_SCALE

            t = torch.randint(0, TRAIN_T_STEPS, (B,), device=device, dtype=torch.long)
            z_noisy, noise_gt = q_sample_train(z0, t)
            noise_pred = cond_unet(z_noisy, t, z_ct)

            loss_noise = F.mse_loss(noise_pred, noise_gt)
            loss = loss_noise

        scaler_ddpm.scale(loss).backward()
        scaler_ddpm.unscale_(optimizer_ddpm)
        nn.utils.clip_grad_norm_(cond_unet.parameters(), 1.0)
        scaler_ddpm.step(optimizer_ddpm)
        scaler_ddpm.update()

        epoch_loss += float(loss.item()) * B
        ns += B
        pbar.set_postfix({'noise_loss': f'{loss_noise.item():.4f}'})

    scheduler_ddpm.step()
    avg_loss = epoch_loss / max(1, ns)
    print(f'[DDPM {epoch+1}] train_noise_loss={avg_loss:.6f}')

    if (epoch + 1) % VAL_EVERY == 0 or epoch == 0:
        val_noise = run_ddpm_val_noise_loss(loader_val, max_batches=VAL_MAX_BATCHES)
        print(f'[DDPM VAL] epoch={epoch+1} noise_loss={val_noise:.6f}')

        if val_noise < best_val_noise:
            best_val_noise = val_noise
            epochs_no_improve = 0
            torch.save(cond_unet.state_dict(), DDPM_BEST)
            print(f'  [BEST] Saved {DDPM_BEST}')
        else:
            epochs_no_improve += 1
            print(f'  [No improve] {epochs_no_improve}/{EARLY_STOP_PATIENCE}')

    torch.save(cond_unet.state_dict(), DDPM_LAST)
    torch.save(
        {
            'epoch': epoch,
            'model': cond_unet.state_dict(),
            'optimizer': optimizer_ddpm.state_dict(),
            'scheduler': scheduler_ddpm.state_dict(),
            'best_val_noise': best_val_noise,
            'epochs_no_improve': epochs_no_improve,
        },
        DDPM_CKPT,
    )

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print('[DDPM EARLY STOP] Validation noise loss plateau reached.')
        break

if os.path.exists(DDPM_BEST):
    cond_unet.load_state_dict(torch.load(DDPM_BEST, map_location=device))
print(f'[OK] DDPM training done. Best val noise loss={best_val_noise:.6f}')

In [ ]:
# =============================================================
# CELL 14 [UPDATED]: DDPM Inference + Full Pipeline Visualization
# Diffusion PSNR/SSIM are reported as secondary quality indicators.
# =============================================================
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torch.nn.functional as F

if 'LATENT_SCALE' not in globals():
    LATENT_SCALE = 0.18215
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

BRIDGE_BEST = os.path.join(OUTPUT_DIR, 'ct2latent_best.pth')
DDPM_BEST = os.path.join(OUTPUT_DIR, 'cond_unet_ddpm_best.pth')
VAE_BEST = os.path.join(OUTPUT_DIR, 'autoencoder_mri_best.pth')
VAE_LAST = os.path.join(OUTPUT_DIR, 'autoencoder_mri.pth')

print('Loading trained models...')
if os.path.exists(DDPM_BEST):
    cond_unet.load_state_dict(torch.load(DDPM_BEST, map_location=device))
    print('[OK] Loaded best DDPM weights.')
else:
    print('[WARN] DDPM best checkpoint not found, using current DDPM weights.')

if os.path.exists(VAE_BEST):
    autoencoder.load_state_dict(torch.load(VAE_BEST, map_location=device))
elif os.path.exists(VAE_LAST):
    autoencoder.load_state_dict(torch.load(VAE_LAST, map_location=device))

if os.path.exists(BRIDGE_BEST):
    ct2latent.load_state_dict(torch.load(BRIDGE_BEST, map_location=device))

autoencoder.eval()
ct2latent.eval()
cond_unet.eval()
print('[OK] All models loaded and set to eval.')


def ddpm_sample(ct_input, n_steps=None, guidance_scale=2.0):
    """Full DDPM reverse diffusion with classifier-free guidance."""
    if n_steps is None:
        n_steps = int(globals().get('T_STEPS', 200))

    B = ct_input.size(0)
    with torch.no_grad():
        z_ct = ct2latent(ct_input) * LATENT_SCALE
        z_uncond = torch.zeros_like(z_ct)
        z = torch.randn(B, 4, 32, 32, device=device)

        for step in reversed(range(n_steps)):
            t_batch = torch.full((B,), step, device=device, dtype=torch.long)
            eps_cond = cond_unet(z, t_batch, z_ct)
            eps_uncond = cond_unet(z, t_batch, z_uncond)
            eps_pred = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

            alpha_t = alphas[step]
            alpha_bar_t = alphas_cumprod[step]
            alpha_bar_prev = alphas_cumprod_prev[step]
            beta_t = betas[step]

            x0_pred = (z - torch.sqrt(1 - alpha_bar_t) * eps_pred) / torch.sqrt(alpha_bar_t)
            x0_pred = torch.clamp(x0_pred, -3, 3)

            coef1 = beta_t * torch.sqrt(alpha_bar_prev) / (1 - alpha_bar_t)
            coef2 = (1 - alpha_bar_prev) * torch.sqrt(alpha_t) / (1 - alpha_bar_t)
            mu = coef1 * x0_pred + coef2 * z

            if step > 0:
                sigma = torch.sqrt(beta_t * (1 - alpha_bar_prev) / (1 - alpha_bar_t))
                z = mu + sigma * torch.randn_like(z)
            else:
                z = mu

        mri_out = autoencoder.decode(z / LATENT_SCALE)
    return torch.clamp(mri_out, 0, 1)


test_loader = val_loader_bridge if 'val_loader_bridge' in globals() else (val_loader if 'val_loader' in globals() else train_loader)
test_batch = next(iter(test_loader))
ct_sample = test_batch['ct'][:4].to(device).float().clamp(0, 1)
mri_gt = test_batch['mri'][:4].to(device).float().clamp(0, 1)

print('Running DDPM reverse diffusion with CFG (guidance_scale=2.0)...')
mri_synth = ddpm_sample(ct_sample, n_steps=min(int(globals().get('T_STEPS', 200)), 100), guidance_scale=2.0)

with torch.no_grad():
    z_gt, _ = autoencoder.encode(mri_gt)
    mri_upper = autoencoder.decode(z_gt).clamp(0, 1)
    z_b = ct2latent(ct_sample)
    mri_bridge = autoencoder.decode(z_b).clamp(0, 1)

# Use shared utilities defined earlier.
psnr_bridge = compute_psnr(mri_bridge, mri_gt)
ssim_bridge = compute_ssim_simple(mri_bridge, mri_gt)
psnr_synth = compute_psnr(mri_synth, mri_gt)
ssim_synth = compute_ssim_simple(mri_synth, mri_gt)
psnr_upper = compute_psnr(mri_upper, mri_gt)

print(f'Bridge   : PSNR={psnr_bridge:.2f} dB | SSIM={ssim_bridge:.4f}')
print(f'Diffusion: PSNR={psnr_synth:.2f} dB | SSIM={ssim_synth:.4f} [secondary]')
print(f'VAE upper: PSNR={psnr_upper:.2f} dB')

fig, axes = plt.subplots(4, 4, figsize=(15, 12))
fig.suptitle(
    f'CT -> MRI via Latent Diffusion (Realism-focused)\n'
    f'Bridge={psnr_bridge:.1f} dB | Diffusion={psnr_synth:.1f} dB (secondary) | VAE={psnr_upper:.1f} dB',
    fontsize=12,
)
for i in range(4):
    axes[0, i].imshow(ct_sample[i, 0].cpu().numpy(), cmap='bone'); axes[0, i].set_title(f'Input CT {i+1}'); axes[0, i].axis('off')
    axes[1, i].imshow(mri_gt[i, 0].cpu().numpy(), cmap='gray'); axes[1, i].set_title(f'Real MRI {i+1}'); axes[1, i].axis('off')
    axes[2, i].imshow(mri_bridge[i, 0].cpu().numpy(), cmap='gray'); axes[2, i].set_title(f'Bridge {i+1}'); axes[2, i].axis('off')
    axes[3, i].imshow(mri_synth[i, 0].cpu().numpy(), cmap='gray'); axes[3, i].set_title(f'Diffusion {i+1}'); axes[3, i].axis('off')

plt.tight_layout()
out_img = os.path.join(OUTPUT_DIR, 'ct_to_mri_results.jpg')
plt.savefig(out_img, dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] Full pipeline visualization saved: {out_img}')

In [ ]:
# CELL 15 [UPDATED]: Quantitative Evaluation + Metrics Logging (shared metrics)
import os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

if 'LATENT_SCALE' not in globals():
    LATENT_SCALE = 0.18215
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

VAE_BEST = os.path.join(OUTPUT_DIR, 'autoencoder_mri_best.pth')
VAE_LAST = os.path.join(OUTPUT_DIR, 'autoencoder_mri.pth')
BRIDGE_BEST = os.path.join(OUTPUT_DIR, 'ct2latent_best.pth')
DDPM_BEST = os.path.join(OUTPUT_DIR, 'cond_unet_ddpm_best.pth')

if os.path.exists(VAE_BEST):
    autoencoder.load_state_dict(torch.load(VAE_BEST, map_location=device))
elif os.path.exists(VAE_LAST):
    autoencoder.load_state_dict(torch.load(VAE_LAST, map_location=device))
if os.path.exists(BRIDGE_BEST):
    ct2latent.load_state_dict(torch.load(BRIDGE_BEST, map_location=device))
if os.path.exists(DDPM_BEST) and 'cond_unet' in globals():
    cond_unet.load_state_dict(torch.load(DDPM_BEST, map_location=device))

autoencoder.eval()
ct2latent.eval()
if 'cond_unet' in globals():
    cond_unet.eval()

@torch.no_grad()
def sample_diffusion_for_eval(ct_batch):
    if 'ddpm_sample_fast' in globals() and os.path.exists(DDPM_BEST):
        return ddpm_sample_fast(ct_batch, n_steps=globals().get('DDPM_SAMPLE_STEPS', 50), guidance_scale=2.0).clamp(0, 1)
    z = ct2latent(ct_batch)
    return autoencoder.decode(z).clamp(0, 1)

loader_eval = val_loader_bridge if 'val_loader_bridge' in globals() else (val_loader if 'val_loader' in globals() else train_loader)
EVAL_MAX_BATCHES = 10

vae_psnr, vae_ssim = [], []
bridge_psnr, bridge_ssim = [], []
diff_psnr, diff_ssim = [], []

with torch.no_grad():
    for bi, batch in enumerate(tqdm(loader_eval, desc='Evaluating metrics')):
        if bi >= EVAL_MAX_BATCHES:
            break

        ct = batch['ct'].to(device).float().clamp(0, 1)
        mri = batch['mri'].to(device).float().clamp(0, 1)

        z_gt, _ = autoencoder.encode(mri)
        mri_vae = autoencoder.decode(z_gt).clamp(0, 1)

        z_bridge = ct2latent(ct)
        mri_bridge = autoencoder.decode(z_bridge).clamp(0, 1)

        mri_diff = sample_diffusion_for_eval(ct)

        vae_psnr.append(compute_psnr(mri_vae, mri))
        vae_ssim.append(compute_ssim_simple(mri_vae, mri))
        bridge_psnr.append(compute_psnr(mri_bridge, mri))
        bridge_ssim.append(compute_ssim_simple(mri_bridge, mri))
        diff_psnr.append(compute_psnr(mri_diff, mri))
        diff_ssim.append(compute_ssim_simple(mri_diff, mri))

summary_rows = [
    {'stage': 'vae_reconstruction', 'psnr_db': round(float(np.mean(vae_psnr)) if vae_psnr else 0.0, 3), 'ssim': round(float(np.mean(vae_ssim)) if vae_ssim else 0.0, 5)},
    {'stage': 'bridge_ct_to_mri', 'psnr_db': round(float(np.mean(bridge_psnr)) if bridge_psnr else 0.0, 3), 'ssim': round(float(np.mean(bridge_ssim)) if bridge_ssim else 0.0, 5)},
    {'stage': 'diffusion_ct_to_mri', 'psnr_db': round(float(np.mean(diff_psnr)) if diff_psnr else 0.0, 3), 'ssim': round(float(np.mean(diff_ssim)) if diff_ssim else 0.0, 5)},
]

metrics_df = pd.DataFrame(summary_rows)
csv_path = os.path.join(OUTPUT_DIR, 'generative_metrics.csv')
metrics_df.to_csv(csv_path, index=False)

print('=' * 65)
print('GENERATION METRICS SUMMARY (shared compute_psnr)')
print('=' * 65)
for row in summary_rows:
    note = ' [secondary for diffusion]' if row['stage'] == 'diffusion_ct_to_mri' else ''
    print(f"{row['stage']:<28} | PSNR={row['psnr_db']:.3f} dB | SSIM={row['ssim']:.5f}{note}")
print('=' * 65)
print(f'Saved: {csv_path}')

In [ ]:
# =============================================================
# CELL 16 [UPDATED]: Stage 4 Setup - Labeled data prep (real + synthetic)
# Adds leakage audit + patient-level split when file paths are available.
# Uses best bridge/diffusion checkpoints for synthetic data generation.
# =============================================================
import os
import re
import random
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset

if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

BRIDGE_BEST = os.path.join(OUTPUT_DIR, 'ct2latent_best.pth')
DDPM_BEST = os.path.join(OUTPUT_DIR, 'cond_unet_ddpm_best.pth')

if os.path.exists(BRIDGE_BEST):
    ct2latent.load_state_dict(torch.load(BRIDGE_BEST, map_location=device))
    print(f'[Classifier Setup] Loaded bridge checkpoint: {BRIDGE_BEST}')
if 'cond_unet' in globals() and os.path.exists(DDPM_BEST):
    cond_unet.load_state_dict(torch.load(DDPM_BEST, map_location=device))
    print(f'[Classifier Setup] Loaded diffusion checkpoint: {DDPM_BEST}')

# If no explicit tumor labels exist, use a deterministic proxy so the pipeline remains runnable.
# Replace this with true clinical labels when available.
def infer_tumor_label_from_mri(mri_tensor):
    high_intensity_ratio = (mri_tensor > 0.75).float().mean().item()
    return 1 if high_intensity_ratio > 0.08 else 0


def extract_patient_id(path_like):
    stem = os.path.splitext(os.path.basename(path_like))[0]
    patterns = [r'(IXI\d+)', r'(BraTS\d+)', r'(patient[_-]?\d+)', r'(subject[_-]?\d+)']
    for p in patterns:
        m = re.search(p, stem, flags=re.IGNORECASE)
        if m:
            return m.group(1).lower()
    # Fallback: take prefix before first separator as pseudo-patient id
    return stem.split('_')[0].split('-')[0].lower()


@torch.no_grad()
def synthesize_from_ct(ct_batch):
    if 'cond_unet' in globals() and 'T_STEPS' in globals() and 'alphas' in globals() and os.path.exists(DDPM_BEST):
        # Diffusion path (trained DDPM available)
        B = ct_batch.size(0)
        z_ct = ct2latent(ct_batch) * LATENT_SCALE
        z = torch.randn(B, 4, 32, 32, device=ct_batch.device)
        n_steps = min(20, int(globals().get('T_STEPS', 50)))
        for step in reversed(range(n_steps)):
            t = torch.full((B,), step, device=ct_batch.device, dtype=torch.long)
            eps = cond_unet(z, t, z_ct)
            a_t = alphas[step]
            a_bar = alphas_cumprod[step]
            z = (1.0 / a_t.sqrt()) * (z - (1 - a_t) / (1 - a_bar).sqrt() * eps)
        out = autoencoder.decode(z / LATENT_SCALE)
        return torch.clamp(out, 0, 1)

    # Bridge-only fallback (raw latent decode, no extra LATENT_SCALE).
    z = ct2latent(ct_batch)
    out = autoencoder.decode(z)
    return torch.clamp(out, 0, 1)


class MRILabeledDataset(Dataset):
    def __init__(self, images, tumor_labels, domain_labels, patient_ids):
        self.images = images
        self.tumor_labels = tumor_labels
        self.domain_labels = domain_labels  # 0 real, 1 synthetic
        self.patient_ids = patient_ids

    def __len__(self):
        return self.images.size(0)

    def __getitem__(self, idx):
        return {
            'mri': self.images[idx],
            'tumor': torch.tensor(self.tumor_labels[idx], dtype=torch.long),
            'domain': torch.tensor(self.domain_labels[idx], dtype=torch.long),
            'patient_id': self.patient_ids[idx],
        }


autoencoder.eval()
ct2latent.eval()
if 'cond_unet' in globals():
    cond_unet.eval()

for p in autoencoder.parameters():
    p.requires_grad = False
for p in ct2latent.parameters():
    p.requires_grad = False
if 'cond_unet' in globals():
    for p in cond_unet.parameters():
        p.requires_grad = False

# Build deterministic source loader (no shuffle) for reproducible split and patient tracing
CLS_BATCH = 32
source_loader = DataLoader(train_ds, batch_size=CLS_BATCH, shuffle=False, num_workers=0, drop_last=False)

# Patient IDs from real dataset paths if available
has_paths = hasattr(train_ds, 'mri_paths') and len(getattr(train_ds, 'mri_paths', [])) == len(train_ds)
if has_paths:
    sample_patient_ids = [extract_patient_id(p) for p in train_ds.mri_paths]
else:
    sample_patient_ids = [f'sample_{i}' for i in range(len(train_ds))]

real_images = []
real_labels = []
real_domains = []
real_patient_ids = []

synth_images = []
synth_labels = []
synth_domains = []
synth_patient_ids = []

sample_cursor = 0
MAX_LABEL_SAMPLES = min(len(train_ds), 1024)

with torch.no_grad():
    for batch in source_loader:
        if sample_cursor >= MAX_LABEL_SAMPLES:
            break

        ct = batch['ct'].to(device)
        mri = batch['mri'].to(device)
        synth = synthesize_from_ct(ct)

        bs = mri.size(0)
        keep = min(bs, MAX_LABEL_SAMPLES - sample_cursor)

        for i in range(keep):
            pid = sample_patient_ids[sample_cursor + i]
            y = infer_tumor_label_from_mri(mri[i])

            real_images.append(mri[i:i+1].cpu())
            real_labels.append(y)
            real_domains.append(0)
            real_patient_ids.append(pid)

            synth_images.append(synth[i:i+1].cpu())
            synth_labels.append(y)
            synth_domains.append(1)
            synth_patient_ids.append(pid)

        sample_cursor += keep

real_mri_all = torch.cat(real_images, dim=0)
synth_mri_all = torch.cat(synth_images, dim=0)

# Leakage audit report
audit_path = os.path.join(OUTPUT_DIR, 'split_leakage_audit.txt')
unique_patients = sorted(set(real_patient_ids))

random.seed(42)
random.shuffle(unique_patients)
split_idx = max(1, int(0.8 * len(unique_patients)))
train_patient_set = set(unique_patients[:split_idx])
val_patient_set = set(unique_patients[split_idx:])

if len(val_patient_set) == 0:
    # fallback if only one pseudo-patient is detected
    val_patient_set = set(list(train_patient_set)[:1])

real_train_idx = [i for i, pid in enumerate(real_patient_ids) if pid in train_patient_set]
real_val_idx = [i for i, pid in enumerate(real_patient_ids) if pid in val_patient_set]

synth_train_idx = [i for i, pid in enumerate(synth_patient_ids) if pid in train_patient_set]

real_ds_full = MRILabeledDataset(real_mri_all, real_labels, real_domains, real_patient_ids)
synth_ds_full = MRILabeledDataset(synth_mri_all, synth_labels, synth_domains, synth_patient_ids)

real_train = Subset(real_ds_full, real_train_idx)
real_val = Subset(real_ds_full, real_val_idx)
synth_train = Subset(synth_ds_full, synth_train_idx)

mix_train = ConcatDataset([real_train, synth_train])

real_train_loader = DataLoader(real_train, batch_size=CLS_BATCH, shuffle=True, num_workers=0)
real_val_loader = DataLoader(real_val, batch_size=CLS_BATCH, shuffle=False, num_workers=0)
mix_train_loader = DataLoader(mix_train, batch_size=CLS_BATCH, shuffle=True, num_workers=0)

overlap = train_patient_set.intersection(val_patient_set)
with open(audit_path, 'w', encoding='utf-8') as f:
    f.write('Leakage Audit\n')
    f.write('============\n')
    f.write(f'Total samples used: {len(real_patient_ids)}\n')
    f.write(f'Total unique patient IDs: {len(unique_patients)}\n')
    f.write(f'Train patient IDs: {len(train_patient_set)}\n')
    f.write(f'Val patient IDs: {len(val_patient_set)}\n')
    f.write(f'Patient overlap count: {len(overlap)}\n')
    f.write(f'Patient overlap IDs: {sorted(list(overlap))}\n')
    f.write(f'Real train samples: {len(real_train_idx)}\n')
    f.write(f'Real val samples: {len(real_val_idx)}\n')
    f.write(f'Synth train samples: {len(synth_train_idx)}\n')

print(f'[OK] Real train samples: {len(real_train)} | Real val samples: {len(real_val)}')
print(f'[OK] Synthetic train samples: {len(synth_train)} | Mixed-train samples: {len(mix_train)}')
print(f'[AUDIT] Leakage report saved: {audit_path}')
print(f'[AUDIT] Patient overlap train-vs-val: {len(overlap)}')
if has_paths:
    print('[AUDIT] Patient IDs inferred from file names.')
else:
    print('[AUDIT] Dataset paths unavailable; using sample-level pseudo IDs (weaker leakage guarantee).')
print('[NOTE] Tumor labels are proxy labels unless you replace infer_tumor_label_from_mri() with real labels.')

In [ ]:
# =============================================================
# CELL 17: Stage 4 Model - ResNet backbone + Tumor head + Domain head (GRL)
# =============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm


class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_grl * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_grl=1.0):
        super().__init__()
        self.lambda_grl = lambda_grl

    def forward(self, x):
        return GradientReversalFn.apply(x, self.lambda_grl)


class DomainAdversarialTumorNet(nn.Module):
    def __init__(self, pretrained=False, lambda_grl=0.5):
        super().__init__()
        weights = tvm.ResNet50_Weights.DEFAULT if pretrained else None
        backbone = tvm.resnet50(weights=weights)

        # Replace first conv for 1-channel MRI
        old_conv = backbone.conv1
        backbone.conv1 = nn.Conv2d(1, old_conv.out_channels, kernel_size=7, stride=2, padding=3, bias=False)

        # Keep feature extractor up to global average pooled feature (2048)
        self.features = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
            backbone.avgpool,
        )
        feat_dim = backbone.fc.in_features

        self.tumor_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 2),
        )

        self.grl = GradientReversalLayer(lambda_grl=lambda_grl)
        self.domain_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 2),
        )

    def set_grl_lambda(self, value):
        self.grl.lambda_grl = float(value)

    def forward(self, x, use_grl=True):
        feat = self.features(x)
        tumor_logits = self.tumor_head(feat)
        if use_grl:
            domain_logits = self.domain_head(self.grl(feat))
        else:
            domain_logits = self.domain_head(feat)
        return tumor_logits, domain_logits


clf_model = DomainAdversarialTumorNet(pretrained=False, lambda_grl=0.5).to(device)
print('[OK] DomainAdversarialTumorNet ready.')
print('Heads: tumor=2-class, domain=2-class, backbone=ResNet50 (1-channel input).')

In [ ]:
# =============================================================
# CELL 18: Stage 4/5 - Train ablations (real-only, +synthetic no GRL, +synthetic with GRL)
# =============================================================
import os
import copy
import numpy as np
import torch
import torch.nn.functional as F

CLF_EPOCHS = 8
LR_CLF = 1e-4
LAMBDA_DOMAIN = 0.3
SAVE_BACKBONE = os.path.join(OUTPUT_DIR, 'tumor_backbone.pth')
SAVE_TUMOR_HEAD = os.path.join(OUTPUT_DIR, 'tumor_classifier_head.pth')
SAVE_DOMAIN_HEAD = os.path.join(OUTPUT_DIR, 'domain_head.pth')


def run_training_experiment(experiment_name, train_loader, use_domain_loss, use_grl):
    model = DomainAdversarialTumorNet(pretrained=False, lambda_grl=LAMBDA_DOMAIN).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_CLF, weight_decay=1e-4)

    history = {'loss': [], 'task': [], 'domain': []}
    best_state = None
    best_val_acc = -1.0

    for epoch in range(CLF_EPOCHS):
        model.train()
        running_loss = 0.0
        running_task = 0.0
        running_dom = 0.0
        n = 0

        for batch in train_loader:
            x = batch['mri'].to(device)
            y_tumor = batch['tumor'].to(device)
            y_domain = batch['domain'].to(device)

            optimizer.zero_grad()
            tumor_logits, domain_logits = model(x, use_grl=use_grl)

            loss_task = F.cross_entropy(tumor_logits, y_tumor)
            if use_domain_loss:
                loss_domain = F.cross_entropy(domain_logits, y_domain)
                loss = loss_task + LAMBDA_DOMAIN * loss_domain
            else:
                loss_domain = torch.tensor(0.0, device=device)
                loss = loss_task

            loss.backward()
            optimizer.step()

            bs = x.size(0)
            n += bs
            running_loss += loss.item() * bs
            running_task += loss_task.item() * bs
            running_dom += loss_domain.item() * bs

        # Validate on real MRI only
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in real_val_loader:
                x = batch['mri'].to(device)
                y = batch['tumor'].to(device)
                logits, _ = model(x, use_grl=False)
                pred = logits.argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.numel()

        val_acc = correct / max(1, total)
        epoch_loss = running_loss / max(1, n)
        epoch_task = running_task / max(1, n)
        epoch_dom = running_dom / max(1, n)

        history['loss'].append(epoch_loss)
        history['task'].append(epoch_task)
        history['domain'].append(epoch_dom)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"[{experiment_name}] Epoch {epoch+1}/{CLF_EPOCHS} | loss={epoch_loss:.4f} task={epoch_task:.4f} domain={epoch_dom:.4f} val_acc(real)={val_acc:.4f}")

    model.load_state_dict(best_state)
    return model, history, best_val_acc


print('Running ablation 1: Real-only baseline...')
model_real_only, hist_real_only, best_real = run_training_experiment(
    experiment_name='RealOnly',
    train_loader=real_train_loader,
    use_domain_loss=False,
    use_grl=False,
)

print('Running ablation 2: Real+Synthetic, no GRL...')
model_syn_no_grl, hist_syn_no_grl, best_syn_no_grl = run_training_experiment(
    experiment_name='Real+Synth-NoGRL',
    train_loader=mix_train_loader,
    use_domain_loss=False,
    use_grl=False,
)

print('Running ablation 3: Real+Synthetic with GRL...')
model_syn_grl, hist_syn_grl, best_syn_grl = run_training_experiment(
    experiment_name='Real+Synth+GRL',
    train_loader=mix_train_loader,
    use_domain_loss=True,
    use_grl=True,
)

# Save the best (GRL) model components as requested
torch.save(model_syn_grl.features.state_dict(), SAVE_BACKBONE)
torch.save(model_syn_grl.tumor_head.state_dict(), SAVE_TUMOR_HEAD)
torch.save(model_syn_grl.domain_head.state_dict(), SAVE_DOMAIN_HEAD)

print('\n[OK] Saved classifier checkpoints:')
print(f' - {SAVE_BACKBONE}')
print(f' - {SAVE_TUMOR_HEAD}')
print(f' - {SAVE_DOMAIN_HEAD}')

ablation_models = {
    'Baseline_RealOnly': model_real_only,
    'PlusSynth_NoGRL': model_syn_no_grl,
    'PlusSynth_WithGRL': model_syn_grl,
}
print('[OK] Ablation models ready for evaluation.')

In [ ]:
# =============================================================
# CELL 19: Stage 5 - Clinical evaluation, ROC, confusion matrices, summary table
# =============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)


def evaluate_classifier(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():
        for batch in loader:
            x = batch['mri'].to(device)
            y = batch['tumor'].to(device)
            logits, _ = model(x, use_grl=False)
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = logits.argmax(dim=1)

            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            y_prob.extend(prob.cpu().numpy().tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / max(1, tp + fn)
    specificity = tn / max(1, tn + fp)

    # Handle edge case where val split has one class only
    if len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_prob)
        fpr, tpr, _ = roc_curve(y_true, y_prob)
    else:
        auc = float('nan')
        fpr, tpr = np.array([0, 1]), np.array([0, 1])

    return {
        'accuracy': float(acc),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'auc': float(auc),
        'cm': np.array([[tn, fp], [fn, tp]]),
        'fpr': fpr,
        'tpr': tpr,
    }


results = {}
for name, model in ablation_models.items():
    results[name] = evaluate_classifier(model, real_val_loader)

summary = pd.DataFrame([
    {
        'Setup': k,
        'Accuracy': round(v['accuracy'], 4),
        'Sensitivity': round(v['sensitivity'], 4),
        'Specificity': round(v['specificity'], 4),
        'ROC_AUC': round(v['auc'], 4) if not np.isnan(v['auc']) else np.nan,
    }
    for k, v in results.items()
])

print('\n=== Tumor Detection (Real-MRI Validation Set) ===')
print(summary)

metrics_csv = os.path.join(OUTPUT_DIR, 'tumor_classifier_ablation_metrics.csv')
summary.to_csv(metrics_csv, index=False)
print(f'\n[OK] Saved summary table: {metrics_csv}')

# ROC Curves
plt.figure(figsize=(8, 6))
for name, res in results.items():
    label = f"{name} (AUC={res['auc']:.3f})" if not np.isnan(res['auc']) else f"{name} (AUC=N/A)"
    plt.plot(res['fpr'], res['tpr'], linewidth=2, label=label)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Tumor Classification Ablations')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
roc_path = os.path.join(OUTPUT_DIR, 'tumor_classifier_roc_curves.png')
plt.savefig(roc_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'[OK] Saved ROC figure: {roc_path}')

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = res['cm']
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75)
cm_path = os.path.join(OUTPUT_DIR, 'tumor_classifier_confusion_matrices.png')
plt.tight_layout()
plt.savefig(cm_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'[OK] Saved confusion matrices: {cm_path}')

# Combined synthesis summary for report-friendly table
synth_metrics_path = os.path.join(OUTPUT_DIR, 'generative_metrics.csv')
if os.path.exists(synth_metrics_path):
    gen_df = pd.read_csv(synth_metrics_path)
    print('\n=== Loaded synthesis metrics (from Cell 15) ===')
    print(gen_df)
else:
    print('\n[WARN] generative_metrics.csv not found yet. Run synthesis metrics cell first if needed.')

In [ ]:
# =============================================================
# CELL 20: Uncertainty report (bootstrap CI) for classifier metrics
# =============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score


def bootstrap_ci(y_true, y_prob, y_pred, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    acc_vals = []
    auc_vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        ypr = y_prob[idx]

        acc_vals.append(accuracy_score(yt, yp))
        if len(np.unique(yt)) > 1:
            auc_vals.append(roc_auc_score(yt, ypr))

    acc_ci = (np.percentile(acc_vals, 2.5), np.percentile(acc_vals, 97.5))
    if len(auc_vals) > 10:
        auc_ci = (np.percentile(auc_vals, 2.5), np.percentile(auc_vals, 97.5))
    else:
        auc_ci = (np.nan, np.nan)
    return acc_ci, auc_ci


def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for batch in loader:
            x = batch['mri'].to(device)
            y = batch['tumor'].to(device)
            logits, _ = model(x, use_grl=False)
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = logits.argmax(dim=1)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            y_prob.extend(prob.cpu().numpy().tolist())
    return np.array(y_true), np.array(y_pred), np.array(y_prob)


ci_rows = []
for name, model in ablation_models.items():
    yt, yp, ypr = collect_predictions(model, real_val_loader)
    acc_ci, auc_ci = bootstrap_ci(yt, ypr, yp, n_boot=1000, seed=42)
    ci_rows.append({
        'Setup': name,
        'Accuracy_CI95_low': round(acc_ci[0], 4),
        'Accuracy_CI95_high': round(acc_ci[1], 4),
        'AUC_CI95_low': round(auc_ci[0], 4) if not np.isnan(auc_ci[0]) else np.nan,
        'AUC_CI95_high': round(auc_ci[1], 4) if not np.isnan(auc_ci[1]) else np.nan,
    })

ci_df = pd.DataFrame(ci_rows)
print('\n=== Bootstrap 95% CI on Real Validation Set ===')
print(ci_df)

ci_csv = os.path.join(OUTPUT_DIR, 'tumor_classifier_bootstrap_ci.csv')
ci_df.to_csv(ci_csv, index=False)
print(f'[OK] Saved CI table: {ci_csv}')